# RunPackageC — EMNIST + COCO Subset Reproduction Notebook

이 notebook은 `RunPackageC_EMNIST_COCO` 실험 패키지의 전체 재현용 버전이다.  
입력 데이터는
- **EMNIST Letters** (symbolic recognition)
- **COCO-derived 4-class subset** (`none / person / car / both`)

으로 구성되며, 두 benchmark에 대해 동일한 phosphene-encoding evaluation pipeline을 적용한다.

주요 특징은 다음과 같다.

1. **benchmark-specific shared decoder**: 각 benchmark마다 clean percept 혼합분포로 decoder를 1회 학습하고, 그 benchmark 내부에서는 encoder, stress, topology, fuzzing 동안 고정한다.  
2. **동일 encoder / 동일 safety proxy / 동일 stress protocol**: rate, sparse, temporal, optim encoder를 동일 규칙 아래 비교한다.  
3. **분리된 저장 경로**: 모든 결과는 `RunPackageC_EMNIST_COCO` 루트 아래에 저장되며, 기존 RunPackageB 계열 결과와 섞이지 않도록 분리된다.  
4. **CPU + Google Colab 기준 실행 가능**: `pilot` / `final` 모드를 분리하고, COCO는 streaming + cached subset manifest 방식으로 처리한다.

권장 실행 순서:
- 처음 1회는 `RUN_MODE="pilot"`로 실행
- 결과가 정상 생성되면 `RUN_MODE="final"`로 전환


In [ ]:
# Cell 0. Install dependencies
import sys, subprocess, pkgutil

REQS = [
    "numpy",
    "pandas",
    "matplotlib",
    "scikit-learn",
    "scipy",
    "pillow",
    "torch",
    "torchvision",
    "tqdm",
    "datasets",
    "ripser",
    "persim",
]

for pkg in REQS:
    mod = pkg.replace("-", "_")
    if mod == "scikit_learn":
        mod = "sklearn"
    if pkgutil.find_loader(mod) is None:
        print(f"[install] {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Dependency setup complete.")


/tmp/ipykernel_58039/1684375931.py:23: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  if pkgutil.find_loader(mod) is None:


[install] pillow
Dependency setup complete.


In [ ]:
# Cell 1. Environment setup, imports, paths, and reproducibility controls
import os
import io
import json
import math
import time
import random
import warnings
import platform
import hashlib
from datetime import datetime
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import defaultdict

# Reproducibility-related environment variables.
# NOTE: PYTHONHASHSEED is most effective when set before the Python process starts;
# it is still recorded here for transparency.
os.environ.setdefault("PYTHONHASHSEED", "42")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageOps
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision.datasets import EMNIST

from scipy.ndimage import gaussian_filter

from datasets import load_dataset

try:
    from ripser import ripser
    from persim import bottleneck
    HAS_RIPSER = True
except Exception as e:
    HAS_RIPSER = False
    print("[warn] ripser/persim unavailable -> topology fallback will be used.", repr(e))

warnings.filterwarnings("ignore", category=UserWarning)

# ---------------------------------------------------------------------
# Global seed and deterministic execution policy
# ---------------------------------------------------------------------
SEED = 42

def seed_everything(seed: int = SEED):
    """Seed Python, NumPy, and PyTorch RNGs for reproducible execution."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Deterministic settings. CPU runs should be deterministic; CUDA can still
    # show small nondeterminism for some kernels, so CPU is preferred for final runs.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:
        torch.use_deterministic_algorithms(True)

def stable_seed(base_seed: int, *parts) -> int:
    """Create a process-independent deterministic integer seed from text parts."""
    msg = "::".join([str(base_seed)] + [str(p) for p in parts])
    digest = hashlib.sha256(msg.encode("utf-8")).hexdigest()
    return int(digest[:8], 16) % (2**31 - 1)

seed_everything(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
print("SEED:", SEED)

# ---------------------------------------------------------------------
# Project root and run-specific output root
# ---------------------------------------------------------------------
IN_COLAB = False
try:
    import google.colab  # type: ignore
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    pass

USE_DRIVE = True if IN_COLAB else False

if USE_DRIVE:
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/RunPackageC_EMNIST_COCO").resolve()
else:
    PROJECT_ROOT = Path("/content/RunPackageC_EMNIST_COCO").resolve()

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

# Shared directories: reused across runs; not manuscript output.
DATA_DIR = PROJECT_ROOT / "data"; DATA_DIR.mkdir(exist_ok=True)
CACHE_DIR = PROJECT_ROOT / "cache"; CACHE_DIR.mkdir(exist_ok=True)

# Output directories: every execution is isolated under a separate run folder.
# To reproduce or compare two runs manually, either leave RUN_ID empty
# for timestamped folders or set os.environ["RUN_ID"] before this cell.
RUN_ID = os.environ.get("RUN_ID", "").strip()
if not RUN_ID:
    RUN_ID = f"Run_seed{SEED:04d}_" + datetime.now().strftime("%Y%m%d_%H%M%S")

RUN_ROOT = PROJECT_ROOT / "runs" / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)

# Backward compatibility: existing functions use ROOT as the output root.
ROOT = RUN_ROOT
SUMMARY_DIR = ROOT / "summary"; SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUN_ROOT:", RUN_ROOT)
print("DATA_DIR:", DATA_DIR)
print("CACHE_DIR:", CACHE_DIR)


DEVICE: cpu
SEED: 42
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT: /content/drive/MyDrive/RunPackageC_EMNIST_COCO
RUN_ROOT: /content/drive/MyDrive/RunPackageC_EMNIST_COCO/runs/Run_seed0042_20260614_003202
DATA_DIR: /content/drive/MyDrive/RunPackageC_EMNIST_COCO/data
CACHE_DIR: /content/drive/MyDrive/RunPackageC_EMNIST_COCO/cache


In [ ]:
# Cell 2. Configuration
@dataclass
class BenchmarkSpec:
    name: str
    n_classes: int
    train_max: int
    val_max: int
    test_max: int
    label_names: tuple

@dataclass
class Cfg:
    run_mode: str = "final"     # "pilot" or "final"
    base_img_size: int = 64
    electrode_grid: int = 16
    percept_size: int = 32
    phosphene_blur_sigma: float = 1.0

    amp_min: float = 0.0
    amp_max: float = 1.0
    f_min: float = 10.0
    f_max: float = 60.0
    pw_min: float = 50e-6
    pw_max: float = 500e-6

    duty_max: float = 0.10
    q_max: float = 0.015
    k_max: int = 40

    stress_levels: tuple = (0.0, 0.25, 0.50, 0.75, 1.00)

    latent_dim: int = 64
    epochs: int = 8
    batch_size: int = 128
    lr: float = 1e-3

    tda_n: int = 300
    tda_pca_dim: int = 10
    tda_maxdim: int = 1

    run_fuzzing: bool = True
    fuzz_seed_images: int = 96
    fuzz_iters: int = 60
    fuzz_mutations_per_iter: int = 12

cfg = Cfg()

if cfg.run_mode == "pilot":
    BENCHMARKS = (
        BenchmarkSpec("emnist_letters", 26, 10000, 2000, 2000, tuple([chr(ord('A') + i) for i in range(26)])),
        BenchmarkSpec("coco_4cls", 4, 3200, 800, 800, ("none", "person", "car", "both")),
    )
    cfg.run_fuzzing = False
    cfg.fuzz_seed_images = 64
    cfg.fuzz_iters = 40
    cfg.fuzz_mutations_per_iter = 10
else:
    BENCHMARKS = (
        BenchmarkSpec("emnist_letters", 26, 20000, 3000, 3000, tuple([chr(ord('A') + i) for i in range(26)])),
        BenchmarkSpec("coco_4cls", 4, 4000, 1000, 1000, ("none", "person", "car", "both")),
    )
    cfg.run_fuzzing = True
    cfg.fuzz_seed_images = 128
    cfg.fuzz_iters = 80
    cfg.fuzz_mutations_per_iter = 16

print("RUN MODE:", cfg.run_mode)
for spec in BENCHMARKS:
    print(spec)


RUN MODE: final
BenchmarkSpec(name='emnist_letters', n_classes=26, train_max=20000, val_max=3000, test_max=3000, label_names=('A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z'))
BenchmarkSpec(name='coco_4cls', n_classes=4, train_max=4000, val_max=1000, test_max=1000, label_names=('none', 'person', 'car', 'both'))


In [ ]:
# Cell 3. Utility functions
# seed_everything() and stable_seed() are defined in Cell 1.

def benchmark_dirs(name: str):
    root = ROOT / name
    fig = root / "figures"; fig.mkdir(parents=True, exist_ok=True)
    tab = root / "tables";  tab.mkdir(parents=True, exist_ok=True)
    cache = root / "cache"; cache.mkdir(parents=True, exist_ok=True)
    model = root / "models"; model.mkdir(parents=True, exist_ok=True)
    raw = root / "raw"; raw.mkdir(parents=True, exist_ok=True)
    return {"root": root, "fig": fig, "tab": tab, "cache": cache, "model": model, "raw": raw}

def stratified_trim_indices(y, n_total, seed=42):
    y = np.asarray(y)
    if n_total is None or n_total >= len(y):
        return np.arange(len(y))
    frac = n_total / len(y)
    idx = []
    rng = np.random.RandomState(seed)
    for c in np.unique(y):
        c_idx = np.where(y == c)[0]
        take = max(1, int(round(len(c_idx) * frac)))
        take = min(take, len(c_idx))
        idx.extend(rng.choice(c_idx, size=take, replace=False).tolist())
    idx = np.array(idx, dtype=int)
    if len(idx) > n_total:
        idx = rng.choice(idx, size=n_total, replace=False)
    elif len(idx) < n_total:
        remain = np.setdiff1d(np.arange(len(y)), idx)
        add = rng.choice(remain, size=n_total-len(idx), replace=False)
        idx = np.concatenate([idx, add])
    rng.shuffle(idx)
    return idx

def pil_to_gray_square(img: Image.Image, size: int = 64, invert: bool = False) -> np.ndarray:
    img = img.convert("L")
    if invert:
        img = ImageOps.invert(img)
    w, h = img.size
    scale = float(size) / max(w, h)
    nw = max(1, int(round(w * scale)))
    nh = max(1, int(round(h * scale)))
    img = img.resize((nw, nh), Image.BILINEAR)
    canvas = Image.new("L", (size, size), color=0)
    left = (size - nw) // 2
    top = (size - nh) // 2
    canvas.paste(img, (left, top))
    arr = np.asarray(canvas, dtype=np.uint8)
    return arr

def ensure_float01(x_uint8: np.ndarray) -> np.ndarray:
    if x_uint8.dtype == np.uint8:
        return x_uint8.astype(np.float32) / 255.0
    return x_uint8.astype(np.float32)

def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False))

def save_run_manifest(extra=None):
    """Save run-level metadata required for reproducibility and manuscript auditing."""
    manifest = {
        "run_id": RUN_ID,
        "seed": SEED,
        "device": DEVICE,
        "project_root": str(PROJECT_ROOT),
        "run_root": str(RUN_ROOT),
        "data_dir": str(DATA_DIR),
        "cache_dir": str(CACHE_DIR),
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "torch": torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "cudnn_deterministic": bool(torch.backends.cudnn.deterministic),
        "cudnn_benchmark": bool(torch.backends.cudnn.benchmark),
        "hashseed_env": os.environ.get("PYTHONHASHSEED", None),
    }
    if extra:
        manifest.update(extra)
    save_json(manifest, SUMMARY_DIR / "RunManifest.json")

def save_cfg_and_benchmarks():
    cfg_dict = asdict(cfg)
    bench_list = [asdict(b) for b in BENCHMARKS]
    save_json({"cfg": cfg_dict, "benchmarks": bench_list}, SUMMARY_DIR / "Config_Benchmarks.json")

def save_split_metadata(spec, data_dict, out_dirs):
    rows = []
    for split_key, label_key in [("train", "y_tr"), ("val", "y_va"), ("test", "y_te")]:
        y = np.asarray(data_dict[label_key])
        vals, counts = np.unique(y, return_counts=True)
        for v, c in zip(vals, counts):
            rows.append({
                "benchmark": spec.name,
                "split": split_key,
                "class_id": int(v),
                "class_name": str(spec.label_names[int(v)]) if int(v) < len(spec.label_names) else str(v),
                "count": int(c),
            })
    pd.DataFrame(rows).to_csv(out_dirs["tab"] / "Table0_SplitClassCounts.csv", index=False)

    # Store labels and shapes as raw experimental metadata without duplicating all cached image arrays.
    np.savez_compressed(
        out_dirs["raw"] / "SplitLabels_And_Shapes.npz",
        y_tr=np.asarray(data_dict["y_tr"]),
        y_va=np.asarray(data_dict["y_va"]),
        y_te=np.asarray(data_dict["y_te"]),
        X_tr_shape=np.asarray(data_dict["X_tr"].shape),
        X_va_shape=np.asarray(data_dict["X_va"].shape),
        X_te_shape=np.asarray(data_dict["X_te"].shape),
    )

def finite_diag(diag: np.ndarray) -> np.ndarray:
    if diag is None or len(diag) == 0:
        return np.zeros((0, 2), dtype=np.float32)
    diag = np.asarray(diag, dtype=np.float32)
    mask = np.isfinite(diag).all(axis=1)
    return diag[mask]

def safe_corr(a, b):
    a = np.asarray(a); b = np.asarray(b)
    if len(a) < 2 or np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

def set_matplotlib_defaults():
    plt.rcParams["figure.dpi"] = 130
    plt.rcParams["savefig.dpi"] = 300
    plt.rcParams["axes.grid"] = True

set_matplotlib_defaults()
save_run_manifest()


## Dataset loaders

이 notebook은 두 benchmark를 모두 다룬다.

- **EMNIST Letters**: `torchvision.datasets.EMNIST(split="letters", download=True)`를 사용한다. Torchvision 문서상 EMNIST는 `letters`를 포함한 6개 split을 제공하며 `download=True`로 자동 다운로드할 수 있다. citeturn383191view0
- **COCO-derived subset**: Hugging Face의 `detection-datasets/coco`를 streaming으로 읽어 class-balanced subset을 구성한다. 이 dataset viewer 기준으로 `train` 117k, `val` 4.95k rows를 가지며 `image`, `image_id`, `objects.category` 등의 필드를 제공한다. citeturn738435view0turn576606search2


In [ ]:
# Cell 4. Dataset loading
COCO_PERSON_ID = 0
COCO_CAR_ID = 2

def emnist_to_numpy(dataset, indices, img_size=64):
    X = np.empty((len(indices), img_size, img_size), dtype=np.uint8)
    y = np.empty((len(indices),), dtype=np.int64)
    for k, idx in enumerate(tqdm(indices, desc="EMNIST materialize", leave=False)):
        img, label = dataset[int(idx)]   # PIL, label
        arr = pil_to_gray_square(img, size=img_size, invert=False)
        X[k] = arr
        y[k] = int(label) - 1            # letters split labels are 1..26
    return X, y

def build_emnist_letters(spec: BenchmarkSpec, seed: int = 42):
    cache_npz = CACHE_DIR / f"{spec.name}_{spec.train_max}_{spec.val_max}_{spec.test_max}_s{seed}.npz"
    if cache_npz.exists():
        z = np.load(cache_npz, allow_pickle=False)
        return {
            "X_tr": z["X_tr"], "y_tr": z["y_tr"],
            "X_va": z["X_va"], "y_va": z["y_va"],
            "X_te": z["X_te"], "y_te": z["y_te"],
            "label_names": np.array(spec.label_names),
        }

    ds_train = EMNIST(root=str(DATA_DIR), split="letters", train=True, download=True)
    ds_test = EMNIST(root=str(DATA_DIR), split="letters", train=False, download=True)

    y_train_full = np.array(ds_train.targets, dtype=np.int64) - 1
    y_test_full = np.array(ds_test.targets, dtype=np.int64) - 1
    idx_all = np.arange(len(y_train_full))

    val_ratio = spec.val_max / max(1, (spec.train_max + spec.val_max))
    idx_tr_full, idx_va_full = train_test_split(
        idx_all, test_size=val_ratio, random_state=seed, stratify=y_train_full
    )

    idx_tr = stratified_trim_indices(y_train_full[idx_tr_full], spec.train_max, seed=seed)
    idx_va = stratified_trim_indices(y_train_full[idx_va_full], spec.val_max, seed=seed)
    idx_te = stratified_trim_indices(y_test_full, spec.test_max, seed=seed)

    idx_tr = idx_tr_full[idx_tr]
    idx_va = idx_va_full[idx_va]

    X_tr, y_tr = emnist_to_numpy(ds_train, idx_tr, img_size=cfg.base_img_size)
    X_va, y_va = emnist_to_numpy(ds_train, idx_va, img_size=cfg.base_img_size)
    X_te, y_te = emnist_to_numpy(ds_test, idx_te, img_size=cfg.base_img_size)

    np.savez_compressed(
        cache_npz, X_tr=X_tr, y_tr=y_tr, X_va=X_va, y_va=y_va, X_te=X_te, y_te=y_te
    )
    return {
        "X_tr": X_tr, "y_tr": y_tr,
        "X_va": X_va, "y_va": y_va,
        "X_te": X_te, "y_te": y_te,
        "label_names": np.array(spec.label_names),
    }

def coco_label_from_example(example) -> int:
    objs = example["objects"]
    if isinstance(objs, dict):
        cats = objs.get("category", [])
    elif isinstance(objs, list):
        cats = [o.get("category") for o in objs]
    else:
        cats = []
    cats = set(int(c) for c in cats)
    has_person = COCO_PERSON_ID in cats
    has_car = COCO_CAR_ID in cats
    if has_person and has_car:
        return 3
    if has_person:
        return 1
    if has_car:
        return 2
    return 0

def preprocess_coco_image(example, img_size=64):
    img = example["image"]
    if not isinstance(img, Image.Image):
        if isinstance(img, dict) and "bytes" in img and img["bytes"] is not None:
            img = Image.open(io.BytesIO(img["bytes"]))
        else:
            raise TypeError("Unsupported COCO image type")
    arr = pil_to_gray_square(img, size=img_size, invert=False)
    return arr

def _balanced_need(counts, target_per_class):
    return any(counts[c] < target_per_class[c] for c in sorted(target_per_class))

def _choose_split_for_label(split_counts, quotas, label):
    for split_name in ("train", "val", "test"):
        if split_counts[split_name][label] < quotas[split_name][label]:
            return split_name
    return None

def build_coco_4cls(spec: BenchmarkSpec, seed: int = 42):
    cache_npz = CACHE_DIR / f"{spec.name}_{spec.train_max}_{spec.val_max}_{spec.test_max}_s{seed}.npz"
    manifest_json = CACHE_DIR / f"{spec.name}_{spec.train_max}_{spec.val_max}_{spec.test_max}_manifest_s{seed}.json"

    if cache_npz.exists():
        z = np.load(cache_npz, allow_pickle=False)
        return {
            "X_tr": z["X_tr"], "y_tr": z["y_tr"],
            "X_va": z["X_va"], "y_va": z["y_va"],
            "X_te": z["X_te"], "y_te": z["y_te"],
            "label_names": np.array(spec.label_names),
        }

    per_class_train = spec.train_max // spec.n_classes
    per_class_val = spec.val_max // spec.n_classes
    per_class_test = spec.test_max // spec.n_classes

    quotas = {
        "train": {c: per_class_train for c in range(spec.n_classes)},
        "val":   {c: per_class_val   for c in range(spec.n_classes)},
        "test":  {c: per_class_test  for c in range(spec.n_classes)},
    }

    split_counts = {
        "train": defaultdict(int),
        "val": defaultdict(int),
        "test": defaultdict(int),
    }

    X_split = {"train": [], "val": [], "test": []}
    y_split = {"train": [], "val": [], "test": []}
    manifest = []

    def all_full():
        return all(
            split_counts[s][c] >= quotas[s][c]
            for s in ("train", "val", "test")
            for c in range(spec.n_classes)
        )

    def consume_stream(split_name_hf: str, local_seed: int):
        ds = load_dataset("detection-datasets/coco", split=split_name_hf, streaming=True)
        ds = ds.shuffle(seed=local_seed, buffer_size=5000)

        pbar = tqdm(ds, desc=f"Build COCO subset from {split_name_hf}", total=None)
        for ex in pbar:
            label = coco_label_from_example(ex)
            split_name = _choose_split_for_label(split_counts, quotas, label)

            if split_name is None:
                if all_full():
                    break
                continue

            arr = preprocess_coco_image(ex, img_size=cfg.base_img_size)
            X_split[split_name].append(arr)
            y_split[split_name].append(label)
            split_counts[split_name][label] += 1

            manifest.append({
                "image_id": int(ex["image_id"]) if "image_id" in ex else None,
                "source_split": split_name_hf,
                "assigned_split": split_name,
                "label": int(label),
            })

            info = " | ".join([
                f"{sp}:{[split_counts[sp][c] for c in range(spec.n_classes)]}"
                for sp in ("train", "val", "test")
            ])
            pbar.set_postfix_str(info)

            if all_full():
                break

    consume_stream("train", seed)

    if not all_full():
        used_extra = False
        for extra_split in ("validation", "val"):
            try:
                consume_stream(extra_split, seed + 1)
                used_extra = True
                if all_full():
                    break
            except Exception as e:
                print(f"[warn] could not use COCO split '{extra_split}': {e}")
        if not used_extra:
            print("[warn] no secondary COCO split could be loaded; continuing with current counts")

    for sp in ("train", "val", "test"):
        got = len(X_split[sp])
        expected = sum(quotas[sp].values())
        if got != expected:
            deficit = expected - got
            per_class_now = [split_counts[sp][c] for c in range(spec.n_classes)]
            raise RuntimeError(
                f"COCO subset build incomplete for {sp}: got {got}, expected {expected}, "
                f"deficit={deficit}, per_class={per_class_now}"
            )

    X_tr = np.stack(X_split["train"], axis=0).astype(np.uint8)
    y_tr = np.array(y_split["train"], dtype=np.int64)
    X_va = np.stack(X_split["val"], axis=0).astype(np.uint8)
    y_va = np.array(y_split["val"], dtype=np.int64)
    X_te = np.stack(X_split["test"], axis=0).astype(np.uint8)
    y_te = np.array(y_split["test"], dtype=np.int64)

    np.savez_compressed(
        cache_npz,
        X_tr=X_tr, y_tr=y_tr,
        X_va=X_va, y_va=y_va,
        X_te=X_te, y_te=y_te
    )
    save_json(manifest, manifest_json)

    return {
        "X_tr": X_tr, "y_tr": y_tr,
        "X_va": X_va, "y_va": y_va,
        "X_te": X_te, "y_te": y_te,
        "label_names": np.array(spec.label_names),
    }

def load_benchmark(spec: BenchmarkSpec, seed: int = 42):
    if spec.name == "emnist_letters":
        return build_emnist_letters(spec, seed=seed)
    elif spec.name == "coco_4cls":
        return build_coco_4cls(spec, seed=seed)
    else:
        raise ValueError(spec.name)


In [ ]:
# Cell 5. Stress model
def _clip01(img):
    return np.clip(img, 0.0, 1.0)

def stress_clean(img, level):
    return img

def stress_gaussian_noise(img, level, sigma_max=0.6):
    sigma = sigma_max * level
    noise = np.random.normal(0.0, sigma, size=img.shape).astype(np.float32)
    return _clip01(img + noise)

def stress_blur(img, level, sigma_max=1.5):
    sigma = sigma_max * level
    if sigma <= 0:
        return img
    return _clip01(gaussian_filter(img, sigma=sigma))

def stress_dropout(img, level):
    p = 0.6 * level
    mask = (np.random.rand(*img.shape) > p).astype(np.float32)
    return _clip01(img * mask)

def stress_occlusion(img, level):
    if level <= 0:
        return img
    h, w = img.shape
    frac = 0.6 * level
    ph, pw = max(1, int(h * frac)), max(1, int(w * frac))
    i0 = np.random.randint(0, h - ph + 1)
    j0 = np.random.randint(0, w - pw + 1)
    out = img.copy()
    out[i0:i0+ph, j0:j0+pw] = 0.0
    return out

STRESS_OPS = {
    "clean": stress_clean,
    "noise": stress_gaussian_noise,
    "blur": stress_blur,
    "dropout": stress_dropout,
    "occlusion": stress_occlusion,
}
LEVELS = list(cfg.stress_levels)
print("Stress ops:", list(STRESS_OPS.keys()))
print("Stress levels:", LEVELS)


Stress ops: ['clean', 'noise', 'blur', 'dropout', 'occlusion']
Stress levels: [0.0, 0.25, 0.5, 0.75, 1.0]


In [ ]:
# Cell 6. Stimulation model, safety constraints, and encoders
@dataclass
class Stimulus:
    amp: np.ndarray
    f_hz: float
    pw_s: float

def project_stimulus_to_bounds(stim: Stimulus) -> Stimulus:
    amp = np.clip(stim.amp, cfg.amp_min, cfg.amp_max)
    f = float(np.clip(stim.f_hz, cfg.f_min, cfg.f_max))
    pw = float(np.clip(stim.pw_s, cfg.pw_min, cfg.pw_max))
    return Stimulus(amp=amp, f_hz=f, pw_s=pw)

def enforce_sparsity(amp: np.ndarray, k_max: int) -> np.ndarray:
    flat = amp.reshape(-1)
    if k_max is None or k_max <= 0:
        return np.zeros_like(amp)
    nz = np.count_nonzero(flat > 1e-12)
    if nz <= k_max:
        return amp
    thr = np.partition(flat, -k_max)[-k_max]
    out = amp.copy()
    out[out < thr] = 0.0
    return out

def compute_constraints(stim: Stimulus):
    duty = float(stim.f_hz * stim.pw_s)
    q_total = float(stim.amp.sum() * stim.f_hz * stim.pw_s)
    active = int(np.count_nonzero(stim.amp > 1e-12))
    return {"duty": duty, "q_total": q_total, "active": active}

def compute_severity(stim: Stimulus):
    c = compute_constraints(stim)
    sev_duty = max(0.0, (c["duty"] - cfg.duty_max) / (cfg.duty_max + 1e-12))
    sev_q = max(0.0, (c["q_total"] - cfg.q_max) / (cfg.q_max + 1e-12))
    sev_k = max(0.0, (c["active"] - cfg.k_max) / (cfg.k_max + 1e-12))
    sev = max(sev_duty, sev_q, sev_k)
    return {
        **c,
        "severity": float(sev),
        "violation": int(sev > 0),
        "sev_duty": float(sev_duty),
        "sev_charge": float(sev_q),
        "sev_sparsity": float(sev_k),
    }

def project_to_safe(stim: Stimulus) -> Stimulus:
    stim = project_stimulus_to_bounds(stim)

    if stim.f_hz * stim.pw_s > cfg.duty_max:
        pw = min(stim.pw_s, cfg.duty_max / (stim.f_hz + 1e-12))
        stim = Stimulus(stim.amp, stim.f_hz, pw)

    amp = enforce_sparsity(stim.amp, cfg.k_max)
    stim = Stimulus(amp, stim.f_hz, stim.pw_s)

    c = compute_constraints(stim)
    if c["q_total"] > cfg.q_max and c["q_total"] > 0:
        scale = cfg.q_max / c["q_total"]
        stim = Stimulus(stim.amp * scale, stim.f_hz, stim.pw_s)

    return project_stimulus_to_bounds(stim)

def resize_to_grid(img: np.ndarray, E: int) -> np.ndarray:
    h, w = img.shape
    ys = np.linspace(0, h - 1, E)
    xs = np.linspace(0, w - 1, E)
    yy, xx = np.meshgrid(ys, xs, indexing="ij")
    y0 = np.floor(yy).astype(int); x0 = np.floor(xx).astype(int)
    y1 = np.clip(y0 + 1, 0, h - 1); x1 = np.clip(x0 + 1, 0, w - 1)
    wy = yy - y0; wx = xx - x0
    Ia = img[y0, x0]; Ib = img[y1, x0]; Ic = img[y0, x1]; Id = img[y1, x1]
    out = (1-wy)*(1-wx)*Ia + wy*(1-wx)*Ib + (1-wy)*wx*Ic + wy*wx*Id
    return out.astype(np.float32)

def phosphene_simulator(stim: Stimulus) -> np.ndarray:
    E = stim.amp.shape[0]
    P = cfg.percept_size
    scale = max(1, int(math.ceil(P / E)))
    amp_up = np.kron(stim.amp, np.ones((scale, scale), dtype=np.float32))
    amp_up = amp_up[:P, :P]
    percept = gaussian_filter(amp_up, sigma=cfg.phosphene_blur_sigma)
    gain = (stim.f_hz / cfg.f_max) * (stim.pw_s / cfg.pw_max)
    percept = np.clip(percept * (0.5 + gain), 0.0, 1.0)
    return percept.astype(np.float32)

def _base_fp(img: np.ndarray):
    m = float(img.mean())
    f = cfg.f_min + (cfg.f_max - cfg.f_min) * m
    pw = min(cfg.pw_max, cfg.duty_max / (f + 1e-12))
    pw = max(cfg.pw_min, pw)
    return f, pw

def greedy_diverse_topk(score_grid: np.ndarray, k: int, min_sep: int = 2):
    E = score_grid.shape[0]
    coords = [(i, j) for i in range(E) for j in range(E)]
    coords = sorted(coords, key=lambda ij: score_grid[ij[0], ij[1]], reverse=True)
    picked = []
    for i, j in coords:
        if len(picked) >= k:
            break
        if all(abs(pi - i) + abs(pj - j) >= min_sep for pi, pj in picked):
            picked.append((i, j))
    return picked

def encoder_rate(img: np.ndarray) -> Stimulus:
    amp = resize_to_grid(img, cfg.electrode_grid)
    amp = cfg.amp_min + (cfg.amp_max - cfg.amp_min) * amp
    f, pw = _base_fp(img)
    return project_to_safe(Stimulus(amp=amp, f_hz=f, pw_s=pw))

def encoder_sparse_diverse(img: np.ndarray) -> Stimulus:
    g = resize_to_grid(img, cfg.electrode_grid)
    picked = greedy_diverse_topk(g, k=cfg.k_max, min_sep=2)
    amp = np.zeros_like(g, dtype=np.float32)
    for i, j in picked:
        amp[i, j] = g[i, j]
    amp = cfg.amp_min + (cfg.amp_max - cfg.amp_min) * amp
    f, pw = _base_fp(img)
    return project_to_safe(Stimulus(amp=amp, f_hz=f, pw_s=pw))

def encoder_temporal_latency(img: np.ndarray, tau_t: float = 0.25) -> Stimulus:
    g = resize_to_grid(img, cfg.electrode_grid)
    flat = g.reshape(-1)
    if np.count_nonzero(flat) > cfg.k_max:
        thr = np.partition(flat, -cfg.k_max)[-cfg.k_max]
    else:
        thr = 0.0
    mask = (g >= thr).astype(np.float32)
    latency = (1.0 - g) * mask
    eff = np.exp(-latency / max(tau_t, 1e-6)) * mask
    eff = eff / (eff.max() + 1e-8)
    amp = 0.8 * cfg.amp_max * eff
    f, pw = _base_fp(img)
    return project_to_safe(Stimulus(amp=amp, f_hz=f, pw_s=pw))

def encoder_optimization(img: np.ndarray, lam: float = 0.08, steps: int = 20, lr: float = 0.5) -> Stimulus:
    target = resize_to_grid(img, cfg.electrode_grid)
    x = target.copy()
    for _ in range(steps):
        x = x - lr * 2.0 * (x - target)
        x = np.sign(x) * np.maximum(np.abs(x) - lam, 0.0)
        x = np.clip(x, 0.0, 1.0)
        x = enforce_sparsity(x, cfg.k_max)
    amp = cfg.amp_min + (cfg.amp_max - cfg.amp_min) * x
    f, pw = _base_fp(img)
    return project_to_safe(Stimulus(amp=amp, f_hz=f, pw_s=pw))

ENCODERS = {
    "rate": encoder_rate,
    "sparse": encoder_sparse_diverse,
    "temporal": encoder_temporal_latency,
    "optim": encoder_optimization,
}
ENCODER_NAMES = list(ENCODERS.keys())
print("Encoders:", ENCODER_NAMES)


Encoders: ['rate', 'sparse', 'temporal', 'optim']


In [ ]:
# Cell 7. Decoder model and clean percept building
class PerceptDataset(Dataset):
    def __init__(self, percepts: np.ndarray, labels: np.ndarray):
        self.X = percepts.astype(np.float32)
        self.y = labels.astype(np.int64)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx][None, :, :]
        y = self.y[idx]
        return torch.from_numpy(x), torch.tensor(y, dtype=torch.long)

class DecoderNet(nn.Module):
    def __init__(self, latent_dim: int, n_classes: int):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        feat_hw = cfg.percept_size // 4
        self.fc1 = nn.Linear(32 * feat_hw * feat_hw, latent_dim)
        self.fc2 = nn.Linear(latent_dim, n_classes)

    def forward(self, x):
        h = self.conv(x)
        h = h.flatten(1)
        z = torch.relu(self.fc1(h))
        logits = self.fc2(z)
        return logits, z

def build_percepts(images_uint8: np.ndarray, encoder_name: str, op_name: str = "clean", level: float = 0.0):
    enc = ENCODERS[encoder_name]
    op = STRESS_OPS[op_name]
    percepts = []
    stims = []
    severities = []
    for img_u8 in images_uint8:
        img = ensure_float01(img_u8)
        img_s = op(img, level)
        stim = enc(img_s)
        sev = compute_severity(stim)
        percept = phosphene_simulator(stim)
        percepts.append(percept)
        stims.append(stim)
        severities.append(sev)
    return np.stack(percepts, axis=0), stims, severities

@torch.no_grad()
def decoder_predict(model: nn.Module, percepts: np.ndarray, batch_size: int = 256):
    preds = []
    zs = []
    model.eval()
    for i in range(0, len(percepts), batch_size):
        xb = torch.from_numpy(percepts[i:i+batch_size, None, :, :]).to(DEVICE)
        logits, z = model(xb)
        preds.append(logits.argmax(dim=1).cpu().numpy())
        zs.append(z.cpu().numpy())
    return np.concatenate(preds), np.concatenate(zs)

def train_shared_decoder(data_dict, n_classes: int, out_model_path: Path, seed: int = SEED):
    seed_everything(seed)
    X_tr = data_dict["X_tr"]; y_tr = data_dict["y_tr"]
    X_va = data_dict["X_va"]; y_va = data_dict["y_va"]

    percepts_tr = []
    labels_tr = []
    percepts_va = []
    labels_va = []

    for en in ENCODER_NAMES:
        P_tr, _, _ = build_percepts(X_tr, en, "clean", 0.0)
        P_va, _, _ = build_percepts(X_va, en, "clean", 0.0)
        percepts_tr.append(P_tr); labels_tr.append(y_tr)
        percepts_va.append(P_va); labels_va.append(y_va)

    percepts_tr = np.concatenate(percepts_tr, axis=0)
    labels_tr = np.concatenate(labels_tr, axis=0)
    percepts_va = np.concatenate(percepts_va, axis=0)
    labels_va = np.concatenate(labels_va, axis=0)

    loader_gen = torch.Generator()
    loader_gen.manual_seed(seed)
    train_loader = DataLoader(
        PerceptDataset(percepts_tr, labels_tr),
        batch_size=cfg.batch_size,
        shuffle=True,
        generator=loader_gen,
        num_workers=0,
    )
    val_loader = DataLoader(
        PerceptDataset(percepts_va, labels_va),
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=0,
    )

    model = DecoderNet(latent_dim=cfg.latent_dim, n_classes=n_classes).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)

    def eval_loader(loader):
        model.eval()
        total = 0
        correct = 0
        loss_sum = 0.0
        with torch.no_grad():
            for xb, yb in loader:
                xb = xb.to(DEVICE); yb = yb.to(DEVICE)
                logits, _ = model(xb)
                loss = F.cross_entropy(logits, yb)
                pred = logits.argmax(dim=1)
                total += int(yb.numel())
                correct += int((pred == yb).sum().item())
                loss_sum += float(loss.item()) * int(yb.numel())
        return {"acc": correct / max(1, total), "loss": loss_sum / max(1, total)}

    hist = []
    for ep in range(1, cfg.epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(DEVICE); yb = yb.to(DEVICE)
            logits, _ = model(xb)
            loss = F.cross_entropy(logits, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
        mtr = eval_loader(train_loader)
        mva = eval_loader(val_loader)
        hist.append({"epoch": ep, "train_acc": mtr["acc"], "val_acc": mva["acc"], "train_loss": mtr["loss"], "val_loss": mva["loss"]})
        print(f"Epoch {ep:02d}/{cfg.epochs} | train acc {mtr['acc']:.3f} | val acc {mva['acc']:.3f}")

    torch.save({"model_state": model.state_dict(), "history": hist, "n_classes": n_classes}, out_model_path)

    for p in model.parameters():
        p.requires_grad = False
    model.eval()
    return model, pd.DataFrame(hist)


In [ ]:
# Cell 8. Core evaluation functions (stress / topology / tri-objective / fuzzing)
def save_method_tables(spec, out_dirs):
    table1 = pd.DataFrame([
        {"Encoder": "rate", "Principle": "rate-based amplitude mapping", "Key knobs": "global f,pw from input statistics; amplitude proportional to intensity"},
        {"Encoder": "sparse", "Principle": "sparse electrode selection with diversity constraint", "Key knobs": "k_max, min_sep; amplitude on selected electrodes"},
        {"Encoder": "temporal", "Principle": "latency modulation plus time-weighted integration", "Key knobs": "tau_t, k_max; latency-weighted amplitude"},
        {"Encoder": "optim", "Principle": "constrained optimization with sparsity regularization", "Key knobs": "lam, steps; projection to feasible space"},
    ])
    table1.to_csv(out_dirs["tab"] / "Table1_EncodingStrategies.csv", index=False)

    table2 = pd.DataFrame([
        {"Constraint": "Duty", "Condition": "f_hz * pw_s <= duty_max", "Severity": "max(0, (duty - duty_max)/duty_max)", "Param": cfg.duty_max},
        {"Constraint": "Charge (proxy)", "Condition": "sum(amp) * f_hz * pw_s <= q_max", "Severity": "max(0, (Q - q_max)/q_max)", "Param": cfg.q_max},
        {"Constraint": "Sparsity", "Condition": "active_electrodes <= k_max", "Severity": "max(0, (active - k_max)/k_max)", "Param": cfg.k_max},
    ])
    table2.to_csv(out_dirs["tab"] / "Table2_SafetyConstraints_SeverityDefinitions.csv", index=False)

    table4_1 = pd.DataFrame([
        {"Category": "Benchmark A", "Item": "EMNIST Letters", "Specification": f"{spec.name == 'emnist_letters'}"},
    ])
    # manuscript-oriented summary
    summary_rows = [
        {"Category": "Benchmark", "Item": "name", "Specification": spec.name},
        {"Category": "Benchmark", "Item": "n_classes", "Specification": spec.n_classes},
        {"Category": "Benchmark", "Item": "split", "Specification": f"{spec.train_max}/{spec.val_max}/{spec.test_max}"},
        {"Category": "Preprocessing", "Item": "input normalization", "Specification": "[0,1] grayscale"},
        {"Category": "Preprocessing", "Item": "electrode grid", "Specification": f"{cfg.electrode_grid}x{cfg.electrode_grid}"},
        {"Category": "Preprocessing", "Item": "percept size", "Specification": f"{cfg.percept_size}x{cfg.percept_size}"},
        {"Category": "Stress", "Item": "operators", "Specification": ", ".join(STRESS_OPS.keys())},
        {"Category": "Stress", "Item": "levels", "Specification": ", ".join([str(v) for v in LEVELS])},
    ]
    pd.DataFrame(summary_rows).to_csv(out_dirs["tab"] / "Table4_1_BenchmarkConfig.csv", index=False)

def save_decoder_table(out_dirs, n_classes):
    table3 = pd.DataFrame([{
        "decoder": "CNN(1-16-32) + FC(latent_dim) + Linear",
        "n_classes": n_classes,
        "percept_size": cfg.percept_size,
        "latent_dim": cfg.latent_dim,
        "epochs": cfg.epochs,
        "batch_size": cfg.batch_size,
        "lr": cfg.lr,
        "training_protocol": "benchmark-specific shared decoder; clean percepts pooled across encoders; trained once then fixed",
    }])
    table3.to_csv(out_dirs["tab"] / "Table3_DecoderConfig.csv", index=False)

def evaluate_stress_sweep(model, data_dict, out_dirs):
    X_te = data_dict["X_te"]; y_te = data_dict["y_te"]

    rows = []
    for encoder_name in ENCODER_NAMES:
        for op_name in STRESS_OPS.keys():
            for level in LEVELS:
                percepts, stims, severities = build_percepts(X_te, encoder_name, op_name, level)
                preds, _ = decoder_predict(model, percepts)
                acc = float((preds == y_te).mean())
                max_sev = float(np.max([d["severity"] for d in severities]))
                mean_sev = float(np.mean([d["severity"] for d in severities]))
                viol_rate = float(np.mean([d["violation"] for d in severities]))
                rows.append({
                    "encoder": encoder_name,
                    "op": op_name,
                    "level": float(level),
                    "accuracy": acc,
                    "max_severity": max_sev,
                    "mean_severity": mean_sev,
                    "violation_rate": viol_rate,
                })

    df = pd.DataFrame(rows)
    df.to_csv(out_dirs["tab"] / "TableStress_PerfSafety_ByEncoderOpLevel.csv", index=False)

    # Table 4 / Table 5
    clean = df[(df.op == "clean") & (df.level == 0.0)].copy()
    clean = clean[["encoder", "accuracy", "max_severity", "mean_severity", "violation_rate"]]
    clean.to_csv(out_dirs["tab"] / "Table4_Clean_PerfSafety.csv", index=False)

    worst_rows = []
    for enc in ENCODER_NAMES:
        d = df[df.encoder == enc].copy()
        r_acc = d.loc[d["accuracy"].idxmin()]
        r_sev = d.loc[d["max_severity"].idxmax()]
        worst_rows.append({
            "encoder": enc,
            "worst_accuracy": float(r_acc["accuracy"]),
            "worst_accuracy_op": r_acc["op"],
            "worst_accuracy_level": float(r_acc["level"]),
            "worst_max_severity": float(r_sev["max_severity"]),
            "worst_severity_op": r_sev["op"],
            "worst_severity_level": float(r_sev["level"]),
        })
    pd.DataFrame(worst_rows).to_csv(out_dirs["tab"] / "Table5_Worst_PerfSafety.csv", index=False)

    # Fig 3: Accuracy vs Level
    plt.figure(figsize=(8, 5))
    for enc in ENCODER_NAMES:
        d = df[(df.encoder == enc) & (df.op != "clean")].copy()
        agg = d.groupby("level", as_index=False)["accuracy"].mean()
        plt.plot(agg["level"], agg["accuracy"], marker="o", label=enc)
    plt.xlabel("Stress level")
    plt.ylabel("Accuracy")
    plt.title("Accuracy vs stress level")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dirs["fig"] / "Fig3_Accuracy_vs_Level.png")
    plt.close()

    # Fig 4: Severity vs Level
    plt.figure(figsize=(8, 5))
    for enc in ENCODER_NAMES:
        d = df[(df.encoder == enc) & (df.op != "clean")].copy()
        agg = d.groupby("level", as_index=False)["max_severity"].mean()
        plt.plot(agg["level"], agg["max_severity"], marker="o", label=enc)
    plt.xlabel("Stress level")
    plt.ylabel("Safety-violation severity")
    plt.title("Severity vs stress level")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dirs["fig"] / "Fig4_Severity_vs_Level.png")
    plt.close()

    return df

def _diag_distance(dgm_a, dgm_b):
    if not HAS_RIPSER:
        return float(np.linalg.norm(np.mean(dgm_a, axis=0) - np.mean(dgm_b, axis=0))) if len(dgm_a) and len(dgm_b) else 0.0
    da = finite_diag(dgm_a)
    db = finite_diag(dgm_b)
    if len(da) == 0 and len(db) == 0:
        return 0.0
    if len(da) == 0:
        da = np.zeros((1, 2), dtype=np.float32)
    if len(db) == 0:
        db = np.zeros((1, 2), dtype=np.float32)
    return float(bottleneck(da, db))

def _compute_diagrams(points: np.ndarray):
    if not HAS_RIPSER:
        return [np.zeros((0, 2), dtype=np.float32), np.zeros((0, 2), dtype=np.float32)]
    result = ripser(points, maxdim=cfg.tda_maxdim)
    dgms = result["dgms"]
    if len(dgms) == 1:
        dgms = [dgms[0], np.zeros((0, 2), dtype=np.float32)]
    return dgms[:2]

def get_representations(model, images_uint8, encoder_name, op_name, level):
    percepts, _, _ = build_percepts(images_uint8, encoder_name, op_name, level)
    _, z = decoder_predict(model, percepts)
    return z

def run_topology_analysis(model, data_dict, out_dirs):
    X_te = data_dict["X_te"]
    n_use = min(cfg.tda_n, len(X_te))
    X_sub = X_te[:n_use]

    rows = []
    diag_cache = {}

    for enc in ENCODER_NAMES:
        Z0 = get_representations(model, X_sub, enc, "clean", 0.0)
        pca = PCA(n_components=min(cfg.tda_pca_dim, Z0.shape[1]), random_state=SEED).fit(Z0)
        P0 = pca.transform(Z0)
        dg0_h0, dg0_h1 = _compute_diagrams(P0)
        diag_cache[(enc, "clean", 0.0)] = (dg0_h0, dg0_h1)

        dists = []
        tmp = []
        for op_name in STRESS_OPS.keys():
            for level in LEVELS:
                Z = get_representations(model, X_sub, enc, op_name, level)
                P = pca.transform(Z)
                dgh0, dgh1 = _compute_diagrams(P)
                diag_cache[(enc, op_name, float(level))] = (dgh0, dgh1)
                dist_h0 = _diag_distance(dg0_h0, dgh0)
                dist_h1 = _diag_distance(dg0_h1, dgh1)
                dist = dist_h0 + dist_h1
                dists.append(dist)
                tmp.append((enc, op_name, float(level), dist_h0, dist_h1, dist))
        tau = np.median([d for d in dists if d > 1e-12]) if np.any(np.array(dists) > 1e-12) else 1.0
        for enc_, op_name, level, dist_h0, dist_h1, dist in tmp:
            tsi = float(np.exp(-dist / max(tau, 1e-8)))
            rows.append({
                "encoder": enc_,
                "op": op_name,
                "level": level,
                "dist_h0": dist_h0,
                "dist_h1": dist_h1,
                "topo_dist": dist,
                "TSI": tsi,
                "tau_scale": tau,
            })

    df_tda = pd.DataFrame(rows)
    df_tda.to_csv(out_dirs["tab"] / "Table8_TDA_TopologyMetrics.csv", index=False)

    # Representative persistence diagrams
    fig, axes = plt.subplots(2, len(ENCODER_NAMES), figsize=(4*len(ENCODER_NAMES), 8))
    for j, enc in enumerate(ENCODER_NAMES):
        axes[0, j].set_title(f"{enc} clean")
        dg0_h0, dg0_h1 = diag_cache[(enc, "clean", 0.0)]
        if len(finite_diag(dg0_h0)):
            axes[0, j].scatter(finite_diag(dg0_h0)[:,0], finite_diag(dg0_h0)[:,1], s=8, label="H0")
        if len(finite_diag(dg0_h1)):
            axes[0, j].scatter(finite_diag(dg0_h1)[:,0], finite_diag(dg0_h1)[:,1], s=8, label="H1")
        axes[0, j].set_xlabel("birth"); axes[0, j].set_ylabel("death")
        d_enc = df_tda[df_tda.encoder == enc].copy()
        d_enc = d_enc[(d_enc.op != "clean")]
        if len(d_enc):
            worst = d_enc.loc[d_enc["topo_dist"].idxmax()]
            dgh0, dgh1 = diag_cache[(enc, worst["op"], float(worst["level"]))]
            axes[1, j].set_title(f"{enc} stressed ({worst['op']}, {worst['level']})")
            if len(finite_diag(dgh0)):
                axes[1, j].scatter(finite_diag(dgh0)[:,0], finite_diag(dgh0)[:,1], s=8, label="H0")
            if len(finite_diag(dgh1)):
                axes[1, j].scatter(finite_diag(dgh1)[:,0], finite_diag(dgh1)[:,1], s=8, label="H1")
            axes[1, j].set_xlabel("birth"); axes[1, j].set_ylabel("death")
    for ax in axes.ravel():
        lim = ax.get_xlim()[1]
        if np.isfinite(lim):
            ax.plot([0, lim], [0, lim], linewidth=1)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="upper right")
    fig.tight_layout()
    fig.savefig(out_dirs["fig"] / "Fig8_Persistence_Diagrams.png")
    plt.close(fig)

    # Fig 9
    plt.figure(figsize=(8, 5))
    for enc in ENCODER_NAMES:
        d = df_tda[(df_tda.encoder == enc) & (df_tda.op != "clean")].copy()
        agg = d.groupby("level", as_index=False)["TSI"].mean()
        plt.plot(agg["level"], agg["TSI"], marker="o", label=enc)
    plt.xlabel("Stress level")
    plt.ylabel("Representation stability (TSI)")
    plt.title("TSI vs stress level")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dirs["fig"] / "Fig9_TSI_vs_Level.png")
    plt.close()

    return df_tda

def run_triobjective(df_stress, df_tda, out_dirs):
    df_join = df_stress.merge(df_tda, on=["encoder","op","level"], how="inner")
    df_nonclean = df_join[df_join.op != "clean"].copy()

    corr_rows = []
    summary_rows = []

    delta = LEVELS[1] - LEVELS[0] if len(LEVELS) > 1 else 1.0
    lam_P, lam_R, lam_S = 1.0, 1.0, 1.0

    for enc in ENCODER_NAMES:
        d = df_nonclean[df_nonclean.encoder == enc].copy()
        if len(d) == 0:
            continue
        corr_rows.append({
            "encoder": enc,
            "corr_TSI_Acc": safe_corr(d["TSI"], d["accuracy"]),
            "corr_TSI_Sev": safe_corr(d["TSI"], d["max_severity"]),
            "corr_Acc_Sev": safe_corr(d["accuracy"], d["max_severity"]),
        })

        tsi_p = float(d.groupby("level")["accuracy"].mean().sum() * delta)
        tsi_r = float(d.groupby("level")["TSI"].mean().sum() * delta)
        safe_risk = float(d.groupby("level")["max_severity"].mean().sum() * delta)
        utility = lam_P * tsi_p + lam_R * tsi_r - lam_S * safe_risk

        summary_rows.append({
            "encoder": enc,
            "min_accuracy": float(d["accuracy"].min()),
            "max_severity": float(d["max_severity"].max()),
            "min_TSI": float(d["TSI"].min()),
            "TSI_P": tsi_p,
            "TSI_R": tsi_r,
            "R_safe": safe_risk,
            "utility": utility,
        })

    df_corr = pd.DataFrame(corr_rows)
    df_corr.to_csv(out_dirs["tab"] / "Table9_Correlations_PerEncoder.csv", index=False)

    df_summary = pd.DataFrame(summary_rows)
    df_summary.to_csv(out_dirs["tab"] / "Table10_TriObjective_Summary.csv", index=False)

    # Fig 10a
    plt.figure(figsize=(6.5, 5))
    for enc in ENCODER_NAMES:
        d = df_nonclean[df_nonclean.encoder == enc]
        plt.scatter(d["TSI"], d["accuracy"], s=18, alpha=0.7, label=enc)
    plt.xlabel("TSI")
    plt.ylabel("Accuracy")
    plt.title("TSI vs Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dirs["fig"] / "Fig10a_TSI_vs_Acc.png")
    plt.close()

    # Fig 10b
    plt.figure(figsize=(6.5, 5))
    for enc in ENCODER_NAMES:
        d = df_nonclean[df_nonclean.encoder == enc]
        plt.scatter(d["TSI"], d["max_severity"], s=18, alpha=0.7, label=enc)
    plt.xlabel("TSI")
    plt.ylabel("Max severity")
    plt.title("TSI vs Severity")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dirs["fig"] / "Fig10b_TSI_vs_Sev.png")
    plt.close()

    # Fig 11
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(111, projection="3d")
    for enc in ENCODER_NAMES:
        d = df_nonclean[df_nonclean.encoder == enc]
        ax.scatter(d["accuracy"], d["max_severity"], d["TSI"], s=18, alpha=0.7, label=enc)
    ax.set_xlabel("Accuracy")
    ax.set_ylabel("Severity")
    ax.set_zlabel("TSI")
    ax.set_title("Tri-objective projection")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_dirs["fig"] / "Fig11_TriObjective_Projection.png")
    plt.close(fig)

    return df_join, df_corr, df_summary

def run_fuzzing(model, data_dict, out_dirs):
    if not cfg.run_fuzzing:
        print("[skip] fuzzing disabled")
        return None, None

    X_te = data_dict["X_te"]; y_te = data_dict["y_te"]

    n_seed = min(cfg.fuzz_seed_images, len(X_te))
    seed_images = X_te[:n_seed]
    seed_labels = y_te[:n_seed]

    cov_rows = []
    counterexamples = []
    coverage = set()

    for enc in ENCODER_NAMES:
        # clean representation reference
        clean_percepts, _, _ = build_percepts(seed_images, enc, "clean", 0.0)
        _, Z0 = decoder_predict(model, clean_percepts)
        pca = PCA(n_components=min(10, Z0.shape[1]), random_state=SEED).fit(Z0)

        for it in range(cfg.fuzz_iters):
            new_bins = 0
            for _ in range(cfg.fuzz_mutations_per_iter):
                idx = np.random.randint(0, n_seed)
                img = ensure_float01(seed_images[idx])
                y = int(seed_labels[idx])

                op = np.random.choice(["noise", "blur", "dropout", "occlusion"])
                level = float(np.random.uniform(0.0, 1.0))
                img_mut = STRESS_OPS[op](img, level)

                stim = ENCODERS[enc](img_mut)
                sev = compute_severity(stim)
                percept = phosphene_simulator(stim)[None, ...]
                pred, z = decoder_predict(model, percept)
                correct = int(pred[0] == y)

                z_ref = pca.transform(Z0[:1])[0]
                z_cur = pca.transform(z)[0]
                rep_dist = float(np.linalg.norm(z_cur - z_ref))

                bin_key = (
                    enc,
                    min(9, int(sev["severity"] * 3)),
                    min(9, int(rep_dist * 2)),
                    correct,
                )
                if bin_key not in coverage:
                    coverage.add(bin_key)
                    new_bins += 1

                score = sev["severity"] + (0 if correct else 1.0) + 0.25 * rep_dist
                counterexamples.append({
                    "encoder": enc,
                    "iter": it,
                    "seed_idx": idx,
                    "op": op,
                    "level": level,
                    "y_true": y,
                    "y_pred": int(pred[0]),
                    "correct": correct,
                    "severity": float(sev["severity"]),
                    "rep_dist": rep_dist,
                    "score": score,
                    "image": (img_mut * 255.0).astype(np.uint8),
                })

            cov_rows.append({"encoder": enc, "iter": it, "coverage_size": len(coverage), "new_bins": new_bins})

    df_cov = pd.DataFrame(cov_rows)
    df_cov.to_csv(out_dirs["tab"] / "Table6_Fuzz_CoverageMetrics.csv", index=False)

    counterexamples = sorted(counterexamples, key=lambda d: d["score"], reverse=True)
    K = min(24, len(counterexamples))
    top = counterexamples[:K]
    df_top = pd.DataFrame([{k: v for k, v in d.items() if k != "image"} for d in top])
    df_top.to_csv(out_dirs["tab"] / "Table7_Fuzz_Counterexamples.csv", index=False)

    # coverage figure
    plt.figure(figsize=(8, 5))
    for enc in ENCODER_NAMES:
        d = df_cov[df_cov.encoder == enc]
        plt.plot(d["iter"], d["coverage_size"], marker="o", label=enc)
    plt.xlabel("Iteration")
    plt.ylabel("Coverage size")
    plt.title("Fuzz coverage growth")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dirs["fig"] / "Fig6_Fuzz_Coverage_Growth.png")
    plt.close()

    # counterexample mosaic
    if len(top):
        cols = 4
        rows = int(np.ceil(len(top) / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(3*cols, 3*rows))
        axes = np.array(axes).reshape(rows, cols)
        for ax in axes.ravel():
            ax.axis("off")
        for ax, d in zip(axes.ravel(), top):
            ax.imshow(d["image"], cmap="gray", vmin=0, vmax=255)
            ax.set_title(f"{d['encoder']} | {d['op']}={d['level']:.2f}\nsev={d['severity']:.2f}, pred={d['y_pred']}")
            ax.axis("off")
        fig.tight_layout()
        fig.savefig(out_dirs["fig"] / "Fig7_Fuzz_Worst_Counterexamples.png")
        plt.close(fig)

    return df_cov, df_top


In [ ]:
# Cell 9. Benchmark runner
def run_benchmark(spec: BenchmarkSpec):
    bench_seed = stable_seed(SEED, spec.name)
    seed_everything(bench_seed)
    print("\n" + "="*90)
    print(f"RUN BENCHMARK: {spec.name}")
    print("="*90)

    out_dirs = benchmark_dirs(spec.name)
    save_method_tables(spec, out_dirs)

    data_dict = load_benchmark(spec, seed=SEED)
    print(
        "Loaded shapes:",
        data_dict["X_tr"].shape, data_dict["y_tr"].shape,
        data_dict["X_va"].shape, data_dict["y_va"].shape,
        data_dict["X_te"].shape, data_dict["y_te"].shape
    )

    save_json({"benchmark": spec.name, "benchmark_seed": int(bench_seed)}, out_dirs["root"] / "BenchmarkSeed.json")
    save_split_metadata(spec, data_dict, out_dirs)

    model_path = out_dirs["model"] / "decoder.pt"
    model, hist = train_shared_decoder(data_dict, spec.n_classes, model_path, seed=bench_seed)
    hist.to_csv(out_dirs["tab"] / "TableDecoder_TrainHistory.csv", index=False)
    save_decoder_table(out_dirs, spec.n_classes)

    # training curves
    plt.figure(figsize=(8, 4.5))
    plt.plot(hist["epoch"], hist["train_acc"], marker="o", label="train_acc")
    plt.plot(hist["epoch"], hist["val_acc"], marker="o", label="val_acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Training curves")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dirs["fig"] / "Fig10_Training_Curves.png")
    plt.close()

    df_stress = evaluate_stress_sweep(model, data_dict, out_dirs)
    df_tda = run_topology_analysis(model, data_dict, out_dirs)
    df_join, df_corr, df_summary = run_triobjective(df_stress, df_tda, out_dirs)
    df_cov, df_top = run_fuzzing(model, data_dict, out_dirs)

    bench_summary = {
        "benchmark": spec.name,
        "n_classes": spec.n_classes,
        "train_n": int(len(data_dict["X_tr"])),
        "val_n": int(len(data_dict["X_va"])),
        "test_n": int(len(data_dict["X_te"])),
        "artifacts_root": str(out_dirs["root"]),
    }
    save_json(bench_summary, out_dirs["root"] / "benchmark_summary.json")
    return {
        "spec": spec,
        "dirs": out_dirs,
        "stress": df_stress,
        "tda": df_tda,
        "join": df_join,
        "corr": df_corr,
        "summary": df_summary,
        "fuzz_cov": df_cov,
        "fuzz_top": df_top,
    }


In [ ]:
# Cell 10. Execute all benchmarks
save_cfg_and_benchmarks()
all_results = {}
t0 = time.time()

for spec in BENCHMARKS:
    all_results[spec.name] = run_benchmark(spec)

elapsed = time.time() - t0
print(f"All benchmarks finished in {elapsed/60:.1f} min")
save_run_manifest({"elapsed_min": elapsed/60.0, "completed": True})



RUN BENCHMARK: emnist_letters
Loaded shapes: (20000, 64, 64) (20000,) (3000, 64, 64) (3000,) (3000, 64, 64) (3000,)
Epoch 01/8 | train acc 0.601 | val acc 0.604
Epoch 02/8 | train acc 0.683 | val acc 0.673
Epoch 03/8 | train acc 0.736 | val acc 0.726
Epoch 04/8 | train acc 0.764 | val acc 0.754
Epoch 05/8 | train acc 0.784 | val acc 0.773
Epoch 06/8 | train acc 0.789 | val acc 0.773
Epoch 07/8 | train acc 0.805 | val acc 0.787
Epoch 08/8 | train acc 0.812 | val acc 0.794

RUN BENCHMARK: coco_4cls
Loaded shapes: (4000, 64, 64) (4000,) (1000, 64, 64) (1000,) (1000, 64, 64) (1000,)
Epoch 01/8 | train acc 0.250 | val acc 0.250
Epoch 02/8 | train acc 0.312 | val acc 0.300
Epoch 03/8 | train acc 0.316 | val acc 0.305
Epoch 04/8 | train acc 0.323 | val acc 0.300
Epoch 05/8 | train acc 0.331 | val acc 0.303
Epoch 06/8 | train acc 0.338 | val acc 0.301
Epoch 07/8 | train acc 0.345 | val acc 0.292
Epoch 08/8 | train acc 0.354 | val acc 0.303
All benchmarks finished in 46.7 min


In [ ]:
# Cell 11. Global sanity check
required_figs = [
    "Fig3_Accuracy_vs_Level.png",
    "Fig4_Severity_vs_Level.png",
    "Fig8_Persistence_Diagrams.png",
    "Fig9_TSI_vs_Level.png",
    "Fig10a_TSI_vs_Acc.png",
    "Fig10b_TSI_vs_Sev.png",
    "Fig11_TriObjective_Projection.png",
]
required_tabs = [
    "Table1_EncodingStrategies.csv",
    "Table2_SafetyConstraints_SeverityDefinitions.csv",
    "Table3_DecoderConfig.csv",
    "Table4_Clean_PerfSafety.csv",
    "Table5_Worst_PerfSafety.csv",
    "Table8_TDA_TopologyMetrics.csv",
    "Table9_Correlations_PerEncoder.csv",
    "Table10_TriObjective_Summary.csv",
]

rows = []
for spec in BENCHMARKS:
    dirs = benchmark_dirs(spec.name)
    for f in required_figs:
        rows.append({"benchmark": spec.name, "kind": "figure", "file": f, "exists": (dirs["fig"] / f).exists()})
    for t in required_tabs:
        rows.append({"benchmark": spec.name, "kind": "table", "file": t, "exists": (dirs["tab"] / t).exists()})

df_check = pd.DataFrame(rows)
df_check.to_csv(SUMMARY_DIR / "SanityCheck_Artifacts.csv", index=False)
print(df_check)


         benchmark    kind                                              file  \
0   emnist_letters  figure                        Fig3_Accuracy_vs_Level.png   
1   emnist_letters  figure                        Fig4_Severity_vs_Level.png   
2   emnist_letters  figure                     Fig8_Persistence_Diagrams.png   
3   emnist_letters  figure                             Fig9_TSI_vs_Level.png   
4   emnist_letters  figure                             Fig10a_TSI_vs_Acc.png   
5   emnist_letters  figure                             Fig10b_TSI_vs_Sev.png   
6   emnist_letters  figure                 Fig11_TriObjective_Projection.png   
7   emnist_letters   table                     Table1_EncodingStrategies.csv   
8   emnist_letters   table  Table2_SafetyConstraints_SeverityDefinitions.csv   
9   emnist_letters   table                          Table3_DecoderConfig.csv   
10  emnist_letters   table                       Table4_Clean_PerfSafety.csv   
11  emnist_letters   table              

In [ ]:
# Cell 12. Compact cross-benchmark summary
summary_rows = []
for spec in BENCHMARKS:
    dirs = benchmark_dirs(spec.name)
    p = dirs["tab"] / "Table10_TriObjective_Summary.csv"
    if p.exists():
        df = pd.read_csv(p)
        df.insert(0, "benchmark", spec.name)
        summary_rows.append(df)

if summary_rows:
    df_all = pd.concat(summary_rows, axis=0, ignore_index=True)
    df_all.to_csv(SUMMARY_DIR / "CrossBenchmark_TriObjective_Summary.csv", index=False)
    print(df_all)
else:
    print("No benchmark summary found.")


        benchmark   encoder  min_accuracy  max_severity   min_TSI     TSI_P  \
0  emnist_letters      rate      0.138333  2.250000e-01  0.006008  0.831833   
1  emnist_letters    sparse      0.143333  2.880891e-07  0.072228  0.765333   
2  emnist_letters  temporal      0.129667  5.400000e+00  0.078202  0.817792   
3  emnist_letters     optim      0.143667  2.250000e-01  0.015841  0.834229   
4       coco_4cls      rate      0.289000  5.000000e-01  0.038506  0.378000   
5       coco_4cls    sparse      0.267000  2.880891e-07  0.149767  0.374250   
6       coco_4cls  temporal      0.278000  5.400000e+00  0.043345  0.379812   
7       coco_4cls     optim      0.259000  5.000000e-01  0.020579  0.372125   

      TSI_R        R_safe   utility  
0  0.642728  1.609375e-01  1.313623  
1  0.649494  2.863817e-07  1.414827  
2  0.615999  4.090625e+00 -2.656834  
3  0.603938  1.437500e-01  1.294417  
4  0.567384  3.156250e-01  0.629759  
5  0.648588  2.863817e-07  1.022838  
6  0.600127  6.015626e

Reviewer 2 response

In [ ]:
# Cell 13
# ============================================================
# Reviewer 2 Comment 1–3 Post-processing Cell
# Colab / Windows compatible
#
# Purpose:
#   Comment 1: COCO diagnostic metrics
#              - accuracy, balanced accuracy, macro-F1
#              - per-class precision/recall/F1
#              - confusion matrices
#
#   Comment 2: Uncertainty estimates
#              - Wilson CI for accuracy
#              - condition-level bootstrap CI for utility
#
#   Comment 3: Pareto dominance summary
#              - three-axis dominance using:
#                TSI_P maximize, TSI_R maximize, R_safe minimize
#
# Output:
#   G:\내 드라이브\RunPackageC_EMNIST_COCO\reviewer2_comment_outputs
#   or, in Colab:
#   /content/drive/MyDrive/RunPackageC_EMNIST_COCO/reviewer2_comment_outputs
# ============================================================

from pathlib import Path
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    balanced_accuracy_score,
    f1_score,
    accuracy_score
)

# ------------------------------------------------------------
# 0. Environment and output-directory setup
# ------------------------------------------------------------

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE_DIR = Path("/content/drive/MyDrive/RunPackageC_EMNIST_COCO")
else:
    BASE_DIR = Path(r"G:\내 드라이브\RunPackageC_EMNIST_COCO")

# If the original notebook already defined ROOT, use it when appropriate.
# This prevents mismatch if the notebook root differs from BASE_DIR.
try:
    if Path(ROOT).exists():
        BASE_DIR = Path(ROOT)
except NameError:
    pass

OUTPUT_BASE = BASE_DIR / "reviewer2_comment_outputs"

COMMENT1_DIR = OUTPUT_BASE / "comment1_coco_diagnostic_metrics"
COMMENT2_DIR = OUTPUT_BASE / "comment2_uncertainty_bootstrap_CI"
COMMENT3_DIR = OUTPUT_BASE / "comment3_pareto_dominance"

for d in [OUTPUT_BASE, COMMENT1_DIR, COMMENT2_DIR, COMMENT3_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_BASE:", OUTPUT_BASE)
print("COMMENT1_DIR:", COMMENT1_DIR)
print("COMMENT2_DIR:", COMMENT2_DIR)
print("COMMENT3_DIR:", COMMENT3_DIR)


# ------------------------------------------------------------
# 1. Sanity checks: required objects from the original notebook
# ------------------------------------------------------------

required_names = [
    "cfg",
    "BENCHMARKS",
    "SEED",
    "DEVICE",
    "ENCODER_NAMES",
    "benchmark_dirs",
    "load_benchmark",
    "DecoderNet",
    "build_percepts",
    "decoder_predict",
]

missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(
        "The following objects are not defined. "
        "Please run the original notebook definition cells first: "
        + ", ".join(missing)
    )

print("\nRequired notebook objects are available.")


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def get_benchmark_spec(name):
    for spec in BENCHMARKS:
        if spec.name == name:
            return spec
    raise RuntimeError(f"Benchmark spec not found: {name}")


def load_frozen_decoder(spec, dirs):
    """
    Load the saved frozen decoder for a benchmark.
    This does not retrain the decoder.
    """
    model_path = dirs["model"] / "decoder.pt"
    if not model_path.exists():
        raise FileNotFoundError(
            f"Saved decoder not found: {model_path}\n"
            "The original benchmark may need to be executed at least once, "
            "or the model file path may be different."
        )

    ckpt = torch.load(model_path, map_location=DEVICE)
    model = DecoderNet(latent_dim=cfg.latent_dim, n_classes=spec.n_classes).to(DEVICE)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    for p in model.parameters():
        p.requires_grad = False

    return model


def wilson_ci(k, n, z=1.96):
    """
    Wilson 95% CI for binomial proportion.
    Used for accuracy CI based on reported accuracy and test-set size.
    """
    if n <= 0:
        return np.nan, np.nan

    p = k / n
    denom = 1.0 + z**2 / n
    center = (p + z**2 / (2*n)) / denom
    half = z * np.sqrt((p * (1-p) + z**2 / (4*n)) / n) / denom

    return center - half, center + half


def percentile_ci(values, alpha=0.05):
    """
    Percentile confidence interval.
    """
    values = np.asarray(values, dtype=float)
    return (
        float(np.percentile(values, 100 * alpha / 2)),
        float(np.percentile(values, 100 * (1 - alpha / 2)))
    )


def safe_round_df(df, cols, ndigits=4):
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = out[c].astype(float).round(ndigits)
    return out


# ============================================================
# Reviewer 2 Comment 1
# COCO clean-condition diagnostic metrics
# ============================================================

print("\n[Comment 1] Generating COCO clean-condition diagnostic metrics...")

coco_spec = get_benchmark_spec("coco_4cls")
coco_dirs = benchmark_dirs("coco_4cls")

# Load data and frozen decoder.
# This reloads the dataset but does not retrain the model.
coco_data = load_benchmark(coco_spec, seed=SEED)
coco_model = load_frozen_decoder(coco_spec, coco_dirs)

X_te = coco_data["X_te"]
y_te = coco_data["y_te"]

class_names = list(coco_spec.label_names)
labels = list(range(coco_spec.n_classes))

coco_summary_rows = []

for enc in ENCODER_NAMES:
    print(f"  Encoder: {enc}")

    # Clean-condition percepts and predictions.
    # This is a post-processing inference step, not model retraining.
    percepts, _, _ = build_percepts(X_te, enc, "clean", 0.0)
    y_pred, _ = decoder_predict(coco_model, percepts)

    y_pred = np.asarray(y_pred)
    y_true = np.asarray(y_te)

    # Save sample-level predictions.
    pred_df = pd.DataFrame({
        "benchmark": "coco_4cls",
        "encoder": enc,
        "sample_index": np.arange(len(y_true)),
        "y_true": y_true,
        "y_pred": y_pred,
        "correct": (y_true == y_pred).astype(int),
    })

    pred_path = COMMENT1_DIR / f"COCO_clean_predictions_{enc}.csv"
    pred_df.to_csv(pred_path, index=False)

    # Confusion matrix.
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    cm_csv_path = COMMENT1_DIR / f"COCO_confusion_matrix_{enc}.csv"
    cm_df.to_csv(cm_csv_path)

    # Confusion matrix figure.
    fig, ax = plt.subplots(figsize=(5.6, 4.8))
    im = ax.imshow(cm)
    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names, rotation=30, ha="right")
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title(f"COCO clean confusion matrix ({enc})")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")

    fig.colorbar(im, ax=ax)
    fig.tight_layout()

    cm_fig_path = COMMENT1_DIR / f"COCO_confusion_matrix_{enc}.png"
    fig.savefig(cm_fig_path, dpi=300)
    plt.close(fig)

    # Per-class precision/recall/F1.
    report = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )
    report_df = pd.DataFrame(report).T
    report_csv_path = COMMENT1_DIR / f"COCO_classification_report_{enc}.csv"
    report_df.to_csv(report_csv_path)

    # Summary metrics.
    acc = float(accuracy_score(y_true, y_pred))
    bacc = float(balanced_accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))

    coco_summary_rows.append({
        "benchmark": "coco_4cls",
        "encoder": enc,
        "n_test": len(y_true),
        "accuracy": acc,
        "balanced_accuracy": bacc,
        "macro_f1": macro_f1,
        "prediction_csv": pred_path.name,
        "confusion_matrix_csv": cm_csv_path.name,
        "classification_report_csv": report_csv_path.name,
        "confusion_matrix_png": cm_fig_path.name,
    })

coco_summary_df = pd.DataFrame(coco_summary_rows)
coco_summary_df = safe_round_df(
    coco_summary_df,
    ["accuracy", "balanced_accuracy", "macro_f1"],
    ndigits=4
)

coco_summary_path = COMMENT1_DIR / "Manuscript_Table_COCO_Diagnostic_Metrics.csv"
coco_summary_df.to_csv(coco_summary_path, index=False)

print("\nCOCO diagnostic metrics saved:")
print(coco_summary_path)
display(coco_summary_df)


# ============================================================
# Reviewer 2 Comment 2
# Accuracy CI and utility condition-level bootstrap CI
# ============================================================

print("\n[Comment 2] Generating uncertainty summaries...")

# ------------------------------------------------------------
# 2A. Accuracy Wilson CI by Encoder × Operator × Level
# ------------------------------------------------------------

accuracy_ci_rows = []

for spec in BENCHMARKS:
    dirs = benchmark_dirs(spec.name)
    stress_path = dirs["tab"] / "TableStress_PerfSafety_ByEncoderOpLevel.csv"

    if not stress_path.exists():
        print(f"  Missing stress table for {spec.name}: {stress_path}")
        continue

    df_stress = pd.read_csv(stress_path)

    summary_path = dirs["root"] / "benchmark_summary.json"
    if summary_path.exists():
        with open(summary_path, "r", encoding="utf-8") as f:
            bench_summary = json.load(f)
        n_test = int(bench_summary.get("test_n", spec.test_max))
    else:
        n_test = int(spec.test_max)

    for _, row in df_stress.iterrows():
        acc = float(row["accuracy"])
        k = int(round(acc * n_test))
        lo, hi = wilson_ci(k, n_test)

        accuracy_ci_rows.append({
            "benchmark": spec.name,
            "encoder": row["encoder"],
            "op": row["op"],
            "level": row["level"],
            "accuracy": acc,
            "n_test": n_test,
            "correct_approx": k,
            "accuracy_CI95_low": lo,
            "accuracy_CI95_high": hi,
            "CI_method": "Wilson binomial CI based on reported accuracy and test-set size",
        })

accuracy_ci_df = pd.DataFrame(accuracy_ci_rows)
accuracy_ci_df = safe_round_df(
    accuracy_ci_df,
    ["accuracy", "accuracy_CI95_low", "accuracy_CI95_high"],
    ndigits=4
)

accuracy_ci_path = COMMENT2_DIR / "Accuracy_Wilson_CI_ByEncoderOpLevel.csv"
accuracy_ci_df.to_csv(accuracy_ci_path, index=False)

print("Accuracy CI table saved:")
print(accuracy_ci_path)


# ------------------------------------------------------------
# 2B. Utility condition-level bootstrap CI
# ------------------------------------------------------------
# This is not repeated-seed retraining.
# It is a condition-level percentile bootstrap over non-clean
# Encoder × Operator × Level rows.
# ------------------------------------------------------------

rng = np.random.default_rng(42)
N_BOOT = 5000

utility_ci_rows = []

for spec in BENCHMARKS:
    dirs = benchmark_dirs(spec.name)
    stress_path = dirs["tab"] / "TableStress_PerfSafety_ByEncoderOpLevel.csv"
    tda_path = dirs["tab"] / "Table8_TDA_TopologyMetrics.csv"

    if not stress_path.exists():
        print(f"  Missing stress table for {spec.name}: {stress_path}")
        continue
    if not tda_path.exists():
        print(f"  Missing TDA table for {spec.name}: {tda_path}")
        continue

    df_stress = pd.read_csv(stress_path)
    df_tda = pd.read_csv(tda_path)

    df_join = df_stress.merge(df_tda, on=["encoder", "op", "level"], how="inner")
    df_nonclean = df_join[df_join["op"] != "clean"].copy()

    for enc in ENCODER_NAMES:
        d = df_nonclean[df_nonclean["encoder"] == enc].copy()
        if len(d) == 0:
            continue

        # Use the same integration logic as Table10:
        # group by stress level, average across operators, integrate over levels.
        levels_sorted = sorted(d["level"].unique())
        if len(levels_sorted) >= 2:
            delta = float(levels_sorted[1] - levels_sorted[0])
        else:
            delta = 1.0

        # Condition-level rows for bootstrap.
        arr = d[["level", "accuracy", "TSI", "max_severity"]].to_numpy()
        n_cond = arr.shape[0]

        boot_util = []
        boot_P = []
        boot_R = []
        boot_S = []

        for _ in range(N_BOOT):
            idx = rng.integers(0, n_cond, size=n_cond)
            sample = pd.DataFrame(
                arr[idx, :],
                columns=["level", "accuracy", "TSI", "max_severity"]
            )

            # Recompute integrated quantities after resampling conditions.
            P = float(sample.groupby("level")["accuracy"].mean().sum() * delta)
            R = float(sample.groupby("level")["TSI"].mean().sum() * delta)
            S = float(sample.groupby("level")["max_severity"].mean().sum() * delta)
            U = P + R - S

            boot_P.append(P)
            boot_R.append(R)
            boot_S.append(S)
            boot_util.append(U)

        # Original non-bootstrap estimate using the same formula.
        P_mean = float(d.groupby("level")["accuracy"].mean().sum() * delta)
        R_mean = float(d.groupby("level")["TSI"].mean().sum() * delta)
        S_mean = float(d.groupby("level")["max_severity"].mean().sum() * delta)
        U_mean = P_mean + R_mean - S_mean

        U_lo, U_hi = percentile_ci(boot_util)

        utility_ci_rows.append({
            "benchmark": spec.name,
            "encoder": enc,
            "n_conditions": n_cond,
            "n_bootstrap": N_BOOT,
            "TSI_P_mean": P_mean,
            "TSI_R_mean": R_mean,
            "R_safe_mean": S_mean,
            "utility_mean": U_mean,
            "utility_CI95_low": U_lo,
            "utility_CI95_high": U_hi,
            "CI_method": "Condition-level percentile bootstrap over non-clean encoder-operator-level rows",
        })

utility_ci_df = pd.DataFrame(utility_ci_rows)
utility_ci_df = safe_round_df(
    utility_ci_df,
    [
        "TSI_P_mean",
        "TSI_R_mean",
        "R_safe_mean",
        "utility_mean",
        "utility_CI95_low",
        "utility_CI95_high",
    ],
    ndigits=4
)

utility_ci_path = COMMENT2_DIR / "Manuscript_Table_Utility_ConditionBootstrap_CI.csv"
utility_ci_df.to_csv(utility_ci_path, index=False)

print("\nUtility condition-level bootstrap CI table saved:")
print(utility_ci_path)
display(utility_ci_df)


# ============================================================
# Reviewer 2 Comment 3
# Pareto dominance summary
# ============================================================

print("\n[Comment 3] Generating Pareto dominance summary...")

pareto_rows = []

for spec in BENCHMARKS:
    dirs = benchmark_dirs(spec.name)
    tri_path = dirs["tab"] / "Table10_TriObjective_Summary.csv"

    if not tri_path.exists():
        print(f"  Missing tri-objective summary for {spec.name}: {tri_path}")
        continue

    df = pd.read_csv(tri_path).copy()

    required_cols = ["encoder", "TSI_P", "TSI_R", "R_safe", "utility"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise RuntimeError(
            f"Missing columns in {tri_path}: {missing_cols}\n"
            f"Available columns: {list(df.columns)}"
        )

    for i, row_i in df.iterrows():
        dominated_by = []
        dominates = []

        for j, row_j in df.iterrows():
            if i == j:
                continue

            # Objective directions:
            # TSI_P: maximize
            # TSI_R: maximize
            # R_safe: minimize
            j_at_least_as_good = (
                (row_j["TSI_P"] >= row_i["TSI_P"]) and
                (row_j["TSI_R"] >= row_i["TSI_R"]) and
                (row_j["R_safe"] <= row_i["R_safe"])
            )
            j_strictly_better = (
                (row_j["TSI_P"] > row_i["TSI_P"]) or
                (row_j["TSI_R"] > row_i["TSI_R"]) or
                (row_j["R_safe"] < row_i["R_safe"])
            )

            if j_at_least_as_good and j_strictly_better:
                dominated_by.append(row_j["encoder"])

            i_at_least_as_good = (
                (row_i["TSI_P"] >= row_j["TSI_P"]) and
                (row_i["TSI_R"] >= row_j["TSI_R"]) and
                (row_i["R_safe"] <= row_j["R_safe"])
            )
            i_strictly_better = (
                (row_i["TSI_P"] > row_j["TSI_P"]) or
                (row_i["TSI_R"] > row_j["TSI_R"]) or
                (row_i["R_safe"] < row_j["R_safe"])
            )

            if i_at_least_as_good and i_strictly_better:
                dominates.append(row_j["encoder"])

        pareto_rows.append({
            "benchmark": spec.name,
            "encoder": row_i["encoder"],
            "TSI_P": row_i["TSI_P"],
            "TSI_R": row_i["TSI_R"],
            "R_safe": row_i["R_safe"],
            "utility": row_i["utility"],
            "non_dominated": len(dominated_by) == 0,
            "dominated_by": ", ".join(dominated_by) if dominated_by else "none",
            "dominates": ", ".join(dominates) if dominates else "none",
        })

pareto_df = pd.DataFrame(pareto_rows)
pareto_df = safe_round_df(
    pareto_df,
    ["TSI_P", "TSI_R", "R_safe", "utility"],
    ndigits=4
)

pareto_path = COMMENT3_DIR / "Manuscript_Table_Pareto_Dominance.csv"
pareto_df.to_csv(pareto_path, index=False)

print("Pareto dominance summary saved:")
print(pareto_path)
display(pareto_df)


# ============================================================
# Final output summary
# ============================================================

print("\n============================================================")
print("Reviewer 2 Comment 1–3 post-processing complete.")
print("All outputs were saved under:")
print(OUTPUT_BASE)
print("============================================================")

print("\nComment 1 outputs:")
for p in sorted(COMMENT1_DIR.glob("*")):
    print(" -", p.name)

print("\nComment 2 outputs:")
for p in sorted(COMMENT2_DIR.glob("*")):
    print(" -", p.name)

print("\nComment 3 outputs:")
for p in sorted(COMMENT3_DIR.glob("*")):
    print(" -", p.name)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE_DIR: /content/drive/MyDrive/RunPackageC_EMNIST_COCO/runs/Run_seed0042_20260614_003202
OUTPUT_BASE: /content/drive/MyDrive/RunPackageC_EMNIST_COCO/runs/Run_seed0042_20260614_003202/reviewer2_comment_outputs
COMMENT1_DIR: /content/drive/MyDrive/RunPackageC_EMNIST_COCO/runs/Run_seed0042_20260614_003202/reviewer2_comment_outputs/comment1_coco_diagnostic_metrics
COMMENT2_DIR: /content/drive/MyDrive/RunPackageC_EMNIST_COCO/runs/Run_seed0042_20260614_003202/reviewer2_comment_outputs/comment2_uncertainty_bootstrap_CI
COMMENT3_DIR: /content/drive/MyDrive/RunPackageC_EMNIST_COCO/runs/Run_seed0042_20260614_003202/reviewer2_comment_outputs/comment3_pareto_dominance

Required notebook objects are available.

[Comment 1] Generating COCO clean-condition diagnostic metrics...
  Encoder: rate
  Encoder: sparse
  Encoder: temporal
  Encoder: optim

COCO diagnostic metrics

,benchmark,encoder,n_test,accuracy,balanced_accuracy,macro_f1,prediction_csv,confusion_matrix_csv,classification_report_csv,confusion_matrix_png
0,coco_4cls,rate,1000,0.301,0.301,0.2932,COCO_clean_predictions_rate.csv,COCO_confusion_matrix_rate.csv,COCO_classification_report_rate.csv,COCO_confusion_matrix_rate.png
1,coco_4cls,sparse,1000,0.300,0.300,0.2892,COCO_clean_predictions_sparse.csv,COCO_confusion_matrix_sparse.csv,COCO_classification_report_sparse.csv,COCO_confusion_matrix_sparse.png
2,coco_4cls,temporal,1000,0.304,0.304,0.2970,COCO_clean_predictions_temporal.csv,COCO_confusion_matrix_temporal.csv,COCO_classification_report_temporal.csv,COCO_confusion_matrix_temporal.png
3,coco_4cls,optim,1000,0.300,0.300,0.2920,COCO_clean_predictions_optim.csv,COCO_confusion_matrix_optim.csv,COCO_classification_report_optim.csv,COCO_confusion_matrix_optim.png



[Comment 2] Generating uncertainty summaries...
Accuracy CI table saved:
/content/drive/MyDrive/RunPackageC_EMNIST_COCO/runs/Run_seed0042_20260614_003202/reviewer2_comment_outputs/comment2_uncertainty_bootstrap_CI/Accuracy_Wilson_CI_ByEncoderOpLevel.csv

Utility condition-level bootstrap CI table saved:
/content/drive/MyDrive/RunPackageC_EMNIST_COCO/runs/Run_seed0042_20260614_003202/reviewer2_comment_outputs/comment2_uncertainty_bootstrap_CI/Manuscript_Table_Utility_ConditionBootstrap_CI.csv


,benchmark,encoder,n_conditions,n_bootstrap,TSI_P_mean,TSI_R_mean,R_safe_mean,utility_mean,utility_CI95_low,utility_CI95_high,CI_method
0,emnist_letters,rate,20,5000,0.8318,0.6427,0.1609,1.3136,1.0366,1.5263,Condition-level percentile bootstrap over non-...
1,emnist_letters,sparse,20,5000,0.7653,0.6495,0.0000,1.4148,1.1112,1.6240,Condition-level percentile bootstrap over non-...
2,emnist_letters,temporal,20,5000,0.8178,0.6160,4.0906,-2.6568,-4.1616,-1.0803,Condition-level percentile bootstrap over non-...
3,emnist_letters,optim,20,5000,0.8342,0.6039,0.1438,1.2944,1.0231,1.4940,Condition-level percentile bootstrap over non-...
4,coco_4cls,rate,20,5000,0.3780,0.5674,0.3156,0.6298,0.4549,0.7686,Condition-level percentile bootstrap over non-...
5,coco_4cls,sparse,20,5000,0.3742,0.6486,0.0000,1.0228,0.8338,1.1278,Condition-level percentile bootstrap over non-...
6,coco_4cls,temporal,20,5000,0.3798,0.6001,0.6016,0.3784,-0.5489,0.7731,Condition-level percentile bootstrap over non-...
7,coco_4cls,optim,20,5000,0.3721,0.5436,0.3125,0.6032,0.4530,0.7322,Condition-level percentile bootstrap over non-...



[Comment 3] Generating Pareto dominance summary...
Pareto dominance summary saved:
/content/drive/MyDrive/RunPackageC_EMNIST_COCO/runs/Run_seed0042_20260614_003202/reviewer2_comment_outputs/comment3_pareto_dominance/Manuscript_Table_Pareto_Dominance.csv


,benchmark,encoder,TSI_P,TSI_R,R_safe,utility,non_dominated,dominated_by,dominates
0,emnist_letters,rate,0.8318,0.6427,0.1609,1.3136,True,none,temporal
1,emnist_letters,sparse,0.7653,0.6495,0.0000,1.4148,True,none,none
2,emnist_letters,temporal,0.8178,0.6160,4.0906,-2.6568,False,rate,none
3,emnist_letters,optim,0.8342,0.6039,0.1438,1.2944,True,none,none
4,coco_4cls,rate,0.3780,0.5674,0.3156,0.6298,True,none,none
5,coco_4cls,sparse,0.3742,0.6486,0.0000,1.0228,True,none,optim
6,coco_4cls,temporal,0.3798,0.6001,0.6016,0.3784,True,none,none
7,coco_4cls,optim,0.3721,0.5436,0.3125,0.6032,False,sparse,none



Reviewer 2 Comment 1–3 post-processing complete.
All outputs were saved under:
/content/drive/MyDrive/RunPackageC_EMNIST_COCO/runs/Run_seed0042_20260614_003202/reviewer2_comment_outputs

Comment 1 outputs:
 - COCO_classification_report_optim.csv
 - COCO_classification_report_rate.csv
 - COCO_classification_report_sparse.csv
 - COCO_classification_report_temporal.csv
 - COCO_clean_predictions_optim.csv
 - COCO_clean_predictions_rate.csv
 - COCO_clean_predictions_sparse.csv
 - COCO_clean_predictions_temporal.csv
 - COCO_confusion_matrix_optim.csv
 - COCO_confusion_matrix_optim.png
 - COCO_confusion_matrix_rate.csv
 - COCO_confusion_matrix_rate.png
 - COCO_confusion_matrix_sparse.csv
 - COCO_confusion_matrix_sparse.png
 - COCO_confusion_matrix_temporal.csv
 - COCO_confusion_matrix_temporal.png
 - Manuscript_Table_COCO_Diagnostic_Metrics.csv

Comment 2 outputs:
 - Accuracy_Wilson_CI_ByEncoderOpLevel.csv
 - Manuscript_Table_Utility_ConditionBootstrap_CI.csv

Comment 3 outputs:
 - Manuscrip

Issue1_2_WeightSensitivity  0612

In [ ]:
# Cell 14
# ============================================================
# Issue 2 Complete Add-on Cell
# Bootstrap CI + Pareto frontier + utility weight sensitivity
#
# Utility convention:
#   U = lambda_P * P + lambda_R * R + lambda_S * S
#
# where:
#   P = TSI_P  : integrated performance score
#   R = TSI_R  : topological representational stability score
#   S = 1 - R_safe : safety score, i.e., complement of residual proxy safety burden
#
# Run this cell AFTER the existing Reviewer 2 post-processing cell.
# ============================================================

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

# ------------------------------------------------------------
# 0. Output directory setup
# ------------------------------------------------------------

IN_COLAB = "google.colab" in sys.modules

try:
    BASE_DIR = Path(ROOT)
except NameError:
    if IN_COLAB:
        BASE_DIR = Path("/content/drive/MyDrive/RunPackageC_EMNIST_COCO")
    else:
        BASE_DIR = Path(r"G:\내 드라이브\RunPackageC_EMNIST_COCO")

OUTPUT_BASE = BASE_DIR / "reviewer2_comment_outputs"

COMMENT1_DIR = OUTPUT_BASE / "comment1_coco_diagnostic_metrics"
COMMENT2_DIR = OUTPUT_BASE / "comment2_uncertainty_bootstrap_CI"
COMMENT3_DIR = OUTPUT_BASE / "comment3_pareto_frontier"
COMMENT4_DIR = OUTPUT_BASE / "comment4_weight_sensitivity"

for d in [COMMENT2_DIR, COMMENT3_DIR, COMMENT4_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_BASE:", OUTPUT_BASE)


# ============================================================
# 1. Sample-level bootstrap CI for COCO clean predictions
# ============================================================

print("\n[Issue 2A] Generating sample-level bootstrap CIs for COCO clean predictions...")

N_BOOT = 5000
RNG_SEED = 2026
rng = np.random.default_rng(RNG_SEED)

def compute_metrics_from_predictions(df):
    y_true = df["y_true"].to_numpy()
    y_pred = df["y_pred"].to_numpy()

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

def stratified_bootstrap_prediction_df(df, rng):
    parts = []
    for _, g in df.groupby("y_true"):
        sampled_idx = rng.choice(g.index.to_numpy(), size=len(g), replace=True)
        parts.append(df.loc[sampled_idx])
    return pd.concat(parts, axis=0, ignore_index=True)

bootstrap_rows = []

pred_files = sorted(COMMENT1_DIR.glob("COCO_clean_predictions_*.csv"))

if len(pred_files) == 0:
    raise FileNotFoundError(
        f"No COCO clean prediction files found in {COMMENT1_DIR}. "
        "Run the existing Reviewer 2 post-processing cell first."
    )

for pred_path in pred_files:
    enc = pred_path.stem.replace("COCO_clean_predictions_", "")
    df_pred = pd.read_csv(pred_path)

    required_cols = {"y_true", "y_pred"}
    if not required_cols.issubset(df_pred.columns):
        raise RuntimeError(
            f"{pred_path.name} does not contain required columns: {required_cols}"
        )

    point = compute_metrics_from_predictions(df_pred)

    boot_values = {
        "accuracy": [],
        "balanced_accuracy": [],
        "macro_f1": [],
    }

    for _ in range(N_BOOT):
        bs = stratified_bootstrap_prediction_df(df_pred, rng)
        m = compute_metrics_from_predictions(bs)
        for k in boot_values:
            boot_values[k].append(m[k])

    for metric_name, values in boot_values.items():
        values = np.asarray(values, dtype=float)
        bootstrap_rows.append({
            "benchmark": "coco_4cls",
            "encoder": enc,
            "metric": metric_name,
            "estimate": point[metric_name],
            "CI95_low": np.percentile(values, 2.5),
            "CI95_high": np.percentile(values, 97.5),
            "n_test": len(df_pred),
            "n_bootstrap": N_BOOT,
            "bootstrap_unit": "test sample",
            "bootstrap_type": "stratified percentile bootstrap by true class",
        })

bootstrap_ci_df = pd.DataFrame(bootstrap_rows)

bootstrap_ci_df_round = bootstrap_ci_df.copy()
for c in ["estimate", "CI95_low", "CI95_high"]:
    bootstrap_ci_df_round[c] = bootstrap_ci_df_round[c].astype(float).round(4)

bootstrap_ci_path = COMMENT2_DIR / "Manuscript_Table_COCO_SampleBootstrap_CI.csv"
bootstrap_ci_df_round.to_csv(bootstrap_ci_path, index=False)

print("Saved:", bootstrap_ci_path)
display(bootstrap_ci_df_round)


# ============================================================
# 2. Pareto frontier analysis from Table10_TriObjective_Summary
# ============================================================

print("\n[Issue 2B] Generating Pareto frontier analysis...")

def is_non_dominated(df, maximize_cols, minimize_cols):
    non_dominated = []

    for i, row_i in df.iterrows():
        dominated = False

        for j, row_j in df.iterrows():
            if i == j:
                continue

            at_least_as_good = True
            strictly_better = False

            for c in maximize_cols:
                if row_j[c] < row_i[c]:
                    at_least_as_good = False
                    break
                if row_j[c] > row_i[c]:
                    strictly_better = True

            if not at_least_as_good:
                continue

            for c in minimize_cols:
                if row_j[c] > row_i[c]:
                    at_least_as_good = False
                    break
                if row_j[c] < row_i[c]:
                    strictly_better = True

            if at_least_as_good and strictly_better:
                dominated = True
                break

        non_dominated.append(not dominated)

    return non_dominated

pareto_rows = []

for spec in BENCHMARKS:
    dirs = benchmark_dirs(spec.name)
    tri_path = dirs["tab"] / "Table10_TriObjective_Summary.csv"

    if not tri_path.exists():
        print(f"Missing tri-objective summary: {tri_path}")
        continue

    df_tri = pd.read_csv(tri_path).copy()

    required_cols = {"encoder", "TSI_P", "TSI_R", "R_safe"}
    if not required_cols.issubset(df_tri.columns):
        raise RuntimeError(
            f"{tri_path} is missing required columns: {required_cols - set(df_tri.columns)}"
        )

    df_tri["benchmark"] = spec.name
    df_tri["non_dominated"] = is_non_dominated(
        df_tri,
        maximize_cols=["TSI_P", "TSI_R"],
        minimize_cols=["R_safe"]
    )

    dominated_by_list = []

    for i, row_i in df_tri.iterrows():
        dominated_by = []

        for j, row_j in df_tri.iterrows():
            if i == j:
                continue

            j_at_least_as_good = (
                (row_j["TSI_P"] >= row_i["TSI_P"]) and
                (row_j["TSI_R"] >= row_i["TSI_R"]) and
                (row_j["R_safe"] <= row_i["R_safe"])
            )
            j_strictly_better = (
                (row_j["TSI_P"] > row_i["TSI_P"]) or
                (row_j["TSI_R"] > row_i["TSI_R"]) or
                (row_j["R_safe"] < row_i["R_safe"])
            )

            if j_at_least_as_good and j_strictly_better:
                dominated_by.append(row_j["encoder"])

        dominated_by_list.append(", ".join(dominated_by) if dominated_by else "none")

    df_tri["dominated_by"] = dominated_by_list

    pareto_rows.append(
        df_tri[[
            "benchmark",
            "encoder",
            "TSI_P",
            "TSI_R",
            "R_safe",
            "non_dominated",
            "dominated_by"
        ]]
    )

pareto_df = pd.concat(pareto_rows, ignore_index=True)

pareto_df_round = pareto_df.copy()
for c in ["TSI_P", "TSI_R", "R_safe"]:
    pareto_df_round[c] = pareto_df_round[c].astype(float).round(4)

pareto_table_path = COMMENT3_DIR / "Manuscript_Table_Pareto_Frontier.csv"
pareto_df_round.to_csv(pareto_table_path, index=False)

print("Saved:", pareto_table_path)
display(pareto_df_round)

# Pareto frontier figure for COCO only.
coco_pareto = pareto_df[pareto_df["benchmark"] == "coco_4cls"].copy()

if len(coco_pareto) > 0:
    fig, ax = plt.subplots(figsize=(6.2, 4.8))

    dominated = coco_pareto[~coco_pareto["non_dominated"]]
    nondom = coco_pareto[coco_pareto["non_dominated"]]

    ax.scatter(
        dominated["R_safe"],
        dominated["TSI_P"],
        marker="o",
        label="Dominated"
    )
    ax.scatter(
        nondom["R_safe"],
        nondom["TSI_P"],
        marker="s",
        label="Pareto frontier"
    )

    for _, row in coco_pareto.iterrows():
        ax.annotate(
            str(row["encoder"]),
            (row["R_safe"], row["TSI_P"]),
            textcoords="offset points",
            xytext=(5, 5),
            fontsize=9
        )

    ax.set_xlabel("Residual proxy safety burden, R_safe (lower is better)")
    ax.set_ylabel("Integrated performance, TSI_P (higher is better)")
    ax.set_title("COCO-derived Pareto frontier projection")
    ax.legend()
    fig.tight_layout()

    pareto_fig_path = COMMENT3_DIR / "Figure_Pareto_Frontier_COCO.png"
    fig.savefig(pareto_fig_path, dpi=600)
    plt.close(fig)

    print("Saved:", pareto_fig_path)


# ============================================================
# 3. Utility weight sensitivity analysis
# ============================================================

print("\n[Issue 2C] Generating utility weight sensitivity analysis...")

weight_scenarios = [
    {
        "scenario": "balanced",
        "lambda_P": 1/3,
        "lambda_R": 1/3,
        "lambda_S": 1/3,
        "interpretation": "equal weighting",
    },
    {
        "scenario": "performance_dominant",
        "lambda_P": 0.6,
        "lambda_R": 0.2,
        "lambda_S": 0.2,
        "interpretation": "performance-prioritized",
    },
    {
        "scenario": "stability_dominant",
        "lambda_P": 0.2,
        "lambda_R": 0.6,
        "lambda_S": 0.2,
        "interpretation": "topological representational stability-prioritized",
    },
    {
        "scenario": "safety_dominant",
        "lambda_P": 0.2,
        "lambda_R": 0.2,
        "lambda_S": 0.6,
        "interpretation": "safety-score-prioritized",
    },
    {
        "scenario": "performance_safety",
        "lambda_P": 0.45,
        "lambda_R": 0.10,
        "lambda_S": 0.45,
        "interpretation": "performance-safety trade-off",
    },
]

weight_rows = []

for spec in BENCHMARKS:
    dirs = benchmark_dirs(spec.name)
    tri_path = dirs["tab"] / "Table10_TriObjective_Summary.csv"

    if not tri_path.exists():
        print(f"Missing tri-objective summary: {tri_path}")
        continue

    df_tri = pd.read_csv(tri_path).copy()

    required_cols = {"encoder", "TSI_P", "TSI_R", "R_safe"}
    if not required_cols.issubset(df_tri.columns):
        raise RuntimeError(
            f"{tri_path} is missing required columns: {required_cols - set(df_tri.columns)}"
        )

    # Convert original tri-objective quantities into manuscript utility components.
    # P: performance score, larger is better.
    # R: topological representational stability score, larger is better.
    # S: safety score, larger is better.
    #    S is defined as the complement of residual proxy safety burden.
    df_tri["P"] = df_tri["TSI_P"]
    df_tri["R"] = df_tri["TSI_R"]
    df_tri["S"] = 1.0 - df_tri["R_safe"]

    for scenario in weight_scenarios:
        lam_P = scenario["lambda_P"]
        lam_R = scenario["lambda_R"]
        lam_S = scenario["lambda_S"]

        temp = df_tri.copy()
        temp["benchmark"] = spec.name
        temp["scenario"] = scenario["scenario"]
        temp["lambda_P"] = lam_P
        temp["lambda_R"] = lam_R
        temp["lambda_S"] = lam_S
        temp["interpretation"] = scenario["interpretation"]

        # Manuscript-consistent utility:
        # U = lambda_P * P + lambda_R * R + lambda_S * S
        temp["weighted_utility"] = (
            lam_P * temp["P"]
            + lam_R * temp["R"]
            + lam_S * temp["S"]
        )

        temp = temp.sort_values("weighted_utility", ascending=False).reset_index(drop=True)
        temp["rank"] = np.arange(1, len(temp) + 1)

        weight_rows.append(
            temp[
                [
                    "benchmark",
                    "scenario",
                    "interpretation",
                    "lambda_P",
                    "lambda_R",
                    "lambda_S",
                    "encoder",
                    "P",
                    "R",
                    "S",
                    "TSI_P",
                    "TSI_R",
                    "R_safe",
                    "weighted_utility",
                    "rank",
                ]
            ]
        )

weight_df = pd.concat(weight_rows, ignore_index=True)

weight_df_round = weight_df.copy()
for c in [
    "lambda_P",
    "lambda_R",
    "lambda_S",
    "P",
    "R",
    "S",
    "TSI_P",
    "TSI_R",
    "R_safe",
    "weighted_utility"
]:
    weight_df_round[c] = weight_df_round[c].astype(float).round(4)

weight_table_path = COMMENT4_DIR / "Manuscript_Table_Utility_WeightSensitivity.csv"
weight_df_round.to_csv(weight_table_path, index=False)

print("Saved:", weight_table_path)
display(weight_df_round)

# Compact rank table for manuscript use.
rank_table = weight_df.pivot_table(
    index=["benchmark", "scenario"],
    columns="rank",
    values="encoder",
    aggfunc="first"
).reset_index()

rank_table.columns = [
    "benchmark",
    "scenario",
] + [f"rank_{int(c)}" for c in rank_table.columns[2:]]

rank_table_path = COMMENT4_DIR / "Manuscript_Table_Utility_WeightSensitivity_RankSummary.csv"
rank_table.to_csv(rank_table_path, index=False)

print("Saved:", rank_table_path)
display(rank_table)

# Figure for COCO weight sensitivity.
coco_weight = weight_df[weight_df["benchmark"] == "coco_4cls"].copy()

if len(coco_weight) > 0:
    scenarios = list(coco_weight["scenario"].unique())
    encoders = list(coco_weight["encoder"].unique())

    x = np.arange(len(scenarios))
    width = 0.8 / max(1, len(encoders))

    fig, ax = plt.subplots(figsize=(8.4, 4.8))

    for k, enc in enumerate(encoders):
        vals = []
        for sc in scenarios:
            d = coco_weight[
                (coco_weight["scenario"] == sc)
                & (coco_weight["encoder"] == enc)
            ]
            vals.append(float(d["weighted_utility"].iloc[0]) if len(d) else np.nan)

        ax.bar(x + k * width, vals, width=width, label=enc)

    ax.set_xticks(x + width * (len(encoders) - 1) / 2)
    ax.set_xticklabels(scenarios, rotation=30, ha="right")
    ax.set_ylabel("Weighted utility, U")
    ax.set_title("COCO-derived utility weight sensitivity")
    ax.legend()
    fig.tight_layout()

    weight_fig_path = COMMENT4_DIR / "Figure_Utility_WeightSensitivity_COCO.png"
    fig.savefig(weight_fig_path, dpi=600)
    plt.close(fig)

    print("Saved:", weight_fig_path)


print("\nIssue 2 complete outputs generated:")
print(" -", bootstrap_ci_path)
print(" -", pareto_table_path)
print(" -", COMMENT3_DIR / "Figure_Pareto_Frontier_COCO.png")
print(" -", weight_table_path)
print(" -", rank_table_path)
print(" -", COMMENT4_DIR / "Figure_Utility_WeightSensitivity_COCO.png")

BASE_DIR: /content/drive/MyDrive/RunPackageC_EMNIST_COCO/runs/Run_seed0042_20260614_003202
OUTPUT_BASE: /content/drive/MyDrive/RunPackageC_EMNIST_COCO/runs/Run_seed0042_20260614_003202/reviewer2_comment_outputs

[Issue 2A] Generating sample-level bootstrap CIs for COCO clean predictions...
Saved: /content/drive/MyDrive/RunPackageC_EMNIST_COCO/runs/Run_seed0042_20260614_003202/reviewer2_comment_outputs/comment2_uncertainty_bootstrap_CI/Manuscript_Table_COCO_SampleBootstrap_CI.csv


,benchmark,encoder,metric,estimate,CI95_low,CI95_high,n_test,n_bootstrap,bootstrap_unit,bootstrap_type
0,coco_4cls,optim,accuracy,0.3000,0.2730,0.3270,1000,5000,test sample,stratified percentile bootstrap by true class
1,coco_4cls,optim,balanced_accuracy,0.3000,0.2730,0.3270,1000,5000,test sample,stratified percentile bootstrap by true class
2,coco_4cls,optim,macro_f1,0.2920,0.2644,0.3199,1000,5000,test sample,stratified percentile bootstrap by true class
3,coco_4cls,rate,accuracy,0.3010,0.2730,0.3290,1000,5000,test sample,stratified percentile bootstrap by true class
4,coco_4cls,rate,balanced_accuracy,0.3010,0.2730,0.3290,1000,5000,test sample,stratified percentile bootstrap by true class
5,coco_4cls,rate,macro_f1,0.2932,0.2655,0.3210,1000,5000,test sample,stratified percentile bootstrap by true class
6,coco_4cls,sparse,accuracy,0.3000,0.2720,0.3280,1000,5000,test sample,stratified percentile bootstrap by true class
7,coco_4cls,sparse,balanced_accuracy,0.3000,0.2720,0.3280,1000,5000,test sample,stratified percentile bootstrap by true class
8,coco_4cls,sparse,macro_f1,0.2892,0.2607,0.3166,1000,5000,test sample,stratified percentile bootstrap by true class
9,coco_4cls,temporal,accuracy,0.3040,0.2760,0.3320,1000,5000,test sample,stratified percentile bootstrap by true class



[Issue 2B] Generating Pareto frontier analysis...
Saved: /content/drive/MyDrive/RunPackageC_EMNIST_COCO/runs/Run_seed0042_20260614_003202/reviewer2_comment_outputs/comment3_pareto_frontier/Manuscript_Table_Pareto_Frontier.csv


,benchmark,encoder,TSI_P,TSI_R,R_safe,non_dominated,dominated_by
0,emnist_letters,rate,0.8318,0.6427,0.1609,True,none
1,emnist_letters,sparse,0.7653,0.6495,0.0000,True,none
2,emnist_letters,temporal,0.8178,0.6160,4.0906,False,rate
3,emnist_letters,optim,0.8342,0.6039,0.1438,True,none
4,coco_4cls,rate,0.3780,0.5674,0.3156,True,none
5,coco_4cls,sparse,0.3742,0.6486,0.0000,True,none
6,coco_4cls,temporal,0.3798,0.6001,0.6016,True,none
7,coco_4cls,optim,0.3721,0.5436,0.3125,False,sparse


Saved: /content/drive/MyDrive/RunPackageC_EMNIST_COCO/runs/Run_seed0042_20260614_003202/reviewer2_comment_outputs/comment3_pareto_frontier/Figure_Pareto_Frontier_COCO.png

[Issue 2C] Generating utility weight sensitivity analysis...
Saved: /content/drive/MyDrive/RunPackageC_EMNIST_COCO/runs/Run_seed0042_20260614_003202/reviewer2_comment_outputs/comment4_weight_sensitivity/Manuscript_Table_Utility_WeightSensitivity.csv


,benchmark,scenario,interpretation,lambda_P,lambda_R,lambda_S,encoder,P,R,S,TSI_P,TSI_R,R_safe,weighted_utility,rank
0,emnist_letters,balanced,equal weighting,0.3333,0.3333,0.3333,sparse,0.7653,0.6495,1.0000,0.7653,0.6495,0.0000,0.8049,1
1,emnist_letters,balanced,equal weighting,0.3333,0.3333,0.3333,rate,0.8318,0.6427,0.8391,0.8318,0.6427,0.1609,0.7712,2
2,emnist_letters,balanced,equal weighting,0.3333,0.3333,0.3333,optim,0.8342,0.6039,0.8562,0.8342,0.6039,0.1438,0.7648,3
3,emnist_letters,balanced,equal weighting,0.3333,0.3333,0.3333,temporal,0.8178,0.6160,-3.0906,0.8178,0.6160,4.0906,-0.5523,4
4,emnist_letters,performance_dominant,performance-prioritized,0.6000,0.2000,0.2000,rate,0.8318,0.6427,0.8391,0.8318,0.6427,0.1609,0.7955,1
5,emnist_letters,performance_dominant,performance-prioritized,0.6000,0.2000,0.2000,optim,0.8342,0.6039,0.8562,0.8342,0.6039,0.1438,0.7926,2
6,emnist_letters,performance_dominant,performance-prioritized,0.6000,0.2000,0.2000,sparse,0.7653,0.6495,1.0000,0.7653,0.6495,0.0000,0.7891,3
7,emnist_letters,performance_dominant,performance-prioritized,0.6000,0.2000,0.2000,temporal,0.8178,0.6160,-3.0906,0.8178,0.6160,4.0906,-0.0043,4
8,emnist_letters,stability_dominant,topological representational stability-priorit...,0.2000,0.6000,0.2000,sparse,0.7653,0.6495,1.0000,0.7653,0.6495,0.0000,0.7428,1
9,emnist_letters,stability_dominant,topological representational stability-priorit...,0.2000,0.6000,0.2000,rate,0.8318,0.6427,0.8391,0.8318,0.6427,0.1609,0.7198,2


Saved: /content/drive/MyDrive/RunPackageC_EMNIST_COCO/runs/Run_seed0042_20260614_003202/reviewer2_comment_outputs/comment4_weight_sensitivity/Manuscript_Table_Utility_WeightSensitivity_RankSummary.csv


,benchmark,scenario,rank_1,rank_2,rank_3,rank_4
0,coco_4cls,balanced,sparse,rate,optim,temporal
1,coco_4cls,performance_dominant,sparse,rate,optim,temporal
2,coco_4cls,performance_safety,sparse,rate,optim,temporal
3,coco_4cls,safety_dominant,sparse,rate,optim,temporal
4,coco_4cls,stability_dominant,sparse,rate,optim,temporal
5,emnist_letters,balanced,sparse,rate,optim,temporal
6,emnist_letters,performance_dominant,rate,optim,sparse,temporal
7,emnist_letters,performance_safety,sparse,optim,rate,temporal
8,emnist_letters,safety_dominant,sparse,optim,rate,temporal
9,emnist_letters,stability_dominant,sparse,rate,optim,temporal


Saved: /content/drive/MyDrive/RunPackageC_EMNIST_COCO/runs/Run_seed0042_20260614_003202/reviewer2_comment_outputs/comment4_weight_sensitivity/Figure_Utility_WeightSensitivity_COCO.png

Issue 2 complete outputs generated:
 - /content/drive/MyDrive/RunPackageC_EMNIST_COCO/runs/Run_seed0042_20260614_003202/reviewer2_comment_outputs/comment2_uncertainty_bootstrap_CI/Manuscript_Table_COCO_SampleBootstrap_CI.csv
 - /content/drive/MyDrive/RunPackageC_EMNIST_COCO/runs/Run_seed0042_20260614_003202/reviewer2_comment_outputs/comment3_pareto_frontier/Manuscript_Table_Pareto_Frontier.csv
 - /content/drive/MyDrive/RunPackageC_EMNIST_COCO/runs/Run_seed0042_20260614_003202/reviewer2_comment_outputs/comment3_pareto_frontier/Figure_Pareto_Frontier_COCO.png
 - /content/drive/MyDrive/RunPackageC_EMNIST_COCO/runs/Run_seed0042_20260614_003202/reviewer2_comment_outputs/comment4_weight_sensitivity/Manuscript_Table_Utility_WeightSensitivity.csv
 - /content/drive/MyDrive/RunPackageC_EMNIST_COCO/runs/Run_seed004